# EpiControl — Continuous Outbreak → DQN Response (No Gradio)

This version keeps the trained V5.6 strategic DQN and the simulation in Python, but uses a standalone **FastAPI + HTML + CSS + JavaScript + Leaflet** web app. It does **not** use Gradio and does **not** retrain the model.

### Intended two-phase flow
1. **START PANDEMIC** resets the scenario and starts an uncontrolled transmission phase. The outbreak build-up uses accelerated movement/transmission to reach a clear citywide outbreak on the map; there are deliberately **no deaths, recoveries, vaccinations, hospital care, quarantine, or DQN actions** in this phase.
2. The spread phase stops automatically when at least 85% of the population has been exposed/infected across at least 80% of zones (or near-total spread is reached). The temporary acceleration is used only during this build-up phase; the original environment parameters are restored for DQN response. A day-limit fallback prevents it from using the entire model episode.
3. **START DQN AGENT** becomes available when the outbreak is established. It takes over the **same environment and preserves the live outbreak state**. The dashboard day remains continuous; the agent’s internal 60-day response clock is rebased at takeover so it gets a full response window. Natural disease outcomes and interventions then resume, so you can see deaths, recoveries, vaccinations, quarantines, hospital use, and actions by the trained policy.

### Run in Google Colab
1. Open this notebook in Colab and choose **Runtime → Run all**.
2. Authorize Google Drive when prompted so the existing V5.6 checkpoint can be loaded.
3. At the end, click **OPEN DASHBOARD IN A NEW TAB**.
4. Click **START PANDEMIC**, let the spread phase finish, then click **START DQN AGENT** once the button becomes available.
5. Keep the Colab runtime and temporary tunnel process alive while using the dashboard.

### What the dashboard includes
- No graphs: a clean, compact control console with a large persistent map, health counters, response capacity, and agent activity.
- Standard OpenStreetMap tiles without a mapping API key.
- Continuous browser-side interpolation of population markers, pulsating infection hotspots, and spreading rings between simulation updates.
- Ambulances animate along a visible route when a successful dispatch event is available, with a direct-line fallback when routing is unavailable.
- The map stays loaded; only simulation layers and live counters update.

**Scientific note:** the model itself still advances in discrete daily simulation steps. The web animations are rendered between these steps to make spread and movement visually continuous. The population, disease, facilities, ambulances, and interventions are synthetic; their positions are visualized within Monaco and are not real epidemiological information. The temporary public tunnel URL is accessible to anyone who has it and works only while the Colab runtime and tunnel are active.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install gymnasium stable-baselines3

import os
import glob
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import (
    EvalCallback,
    CheckpointCallback,
    CallbackList
)
from stable_baselines3.common.env_checker import check_env


warnings.filterwarnings(
    "ignore",
    message=r".*Kernel\._parent_header is deprecated.*"
)
warnings.filterwarnings(
    "ignore",
    message=r".*datetime\.datetime\.utcnow\(\) is deprecated.*"
)

print("Environment setup complete.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 4.3 MB/s eta 0:00:00
Environment setup complete.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

POPULATION = 1000
CITY_SIZE = 100.0

INITIAL_INFECTED = 5

INFECTION_RADIUS = 4.0
TRANSMISSION_PROBABILITY = 0.30
DISTANCE_SCALE = 2.0
VACCINE_EFFECTIVENESS = 0.85

INCUBATION_DAYS = 4
INFECTIOUS_DAYS = 10

BASE_MOVEMENT = 1.0

GRID_SIZE = 5
NUM_ZONES = GRID_SIZE ** 2
ZONE_SIZE = CITY_SIZE / GRID_SIZE

NUM_AMBULANCES = 5

# The two hospitals below provide 40 + 60 beds.
HOSPITAL_CAPACITY = 100

INITIAL_VACCINES = 150
INITIAL_TESTS = 500
INITIAL_BUDGET = 10_000

SIMULATION_DAYS = 60

# Resource costs used by the RL environment.
RESOURCE_COSTS = {
    "vaccination": 5,
    "open_shelter": 500,
    "quarantine_person": 2,
    "ambulance_dispatch": 100,
    "test": 3
}

print("Configuration ready.")

Configuration ready.


In [ ]:
#HEALTH STATUS
SUSCEPTIBLE = 0
EXPOSED = 1
INFECTED = 2
RECOVERED = 3
DEAD = 4

STATE_NAMES = {
    SUSCEPTIBLE: "Susceptible",
    EXPOSED: "Exposed",
    INFECTED: "Infected",
    RECOVERED: "Recovered",
    DEAD: "Dead"
}

STATE_COLORS = {
    SUSCEPTIBLE: "green",
    EXPOSED: "gold",
    INFECTED: "red",
    RECOVERED: "blue",
    DEAD: "black"
}

# Agent action types
NO_ACTION = "NO_ACTION"
VACCINATE = "VACCINATE"
OPEN_SHELTER = "OPEN_SHELTER"
QUARANTINE = "QUARANTINE"
DISPATCH_AMBULANCE = "DISPATCH_AMBULANCE"

VALID_ACTIONS = {
    NO_ACTION,
    VACCINATE,
    OPEN_SHELTER,
    QUARANTINE,
    DISPATCH_AMBULANCE
}

print("State and action definitions ready.")

State and action definitions ready.


In [ ]:
# POPULATION GENERATION
def generate_population(seed=SEED):
    local_rng = np.random.default_rng(seed)

    positions = local_rng.uniform(
        0,
        CITY_SIZE,
        size=(POPULATION, 2)
    )

    ages = local_rng.integers(
        1,
        91,
        size=POPULATION
    )

    mobility = local_rng.uniform(
        0.5,
        1.5,
        size=POPULATION
    )

    states = np.full(
        POPULATION,
        SUSCEPTIBLE,
        dtype=np.int8
    )

    exposure_day = np.full(
        POPULATION,
        -1,
        dtype=np.int16
    )

    needs_hospital = np.zeros(
        POPULATION,
        dtype=bool
    )

    hospitalized = np.zeros(
        POPULATION,
        dtype=bool
    )

    center = np.array([
        CITY_SIZE / 2,
        CITY_SIZE / 2
    ])

    distances = np.linalg.norm(
        positions - center,
        axis=1
    )

    initial_indices = np.argsort(
        distances
    )[:INITIAL_INFECTED]

    states[initial_indices] = INFECTED
    exposure_day[initial_indices] = -INCUBATION_DAYS

    return (
        positions,
        ages,
        mobility,
        states,
        exposure_day,
        needs_hospital,
        hospitalized
    )

(
    INITIAL_POSITIONS,
    INITIAL_AGES,
    INITIAL_MOBILITY,
    INITIAL_STATES,
    INITIAL_EXPOSURE_DAY,
    INITIAL_NEEDS_HOSPITAL,
    INITIAL_HOSPITALIZED
) = generate_population()

print("Population:", POPULATION)
print("Initial infected:", np.sum(INITIAL_STATES == INFECTED))

Population: 1000
Initial infected: 5


In [ ]:
# DISEASE AND MOBILITY FUNCTIONS

def get_zone_id(x, y):
    col = int(x / ZONE_SIZE)
    row = int(y / ZONE_SIZE)
    col = np.clip(col, 0, GRID_SIZE - 1)
    row = np.clip(row, 0, GRID_SIZE - 1)
    return row * GRID_SIZE + col


def update_person_zones(positions):
    return np.array([
        get_zone_id(x, y)
        for x, y in positions
    ], dtype=np.int16)


def move_population(
    positions,
    states,
    hospitalized,
    quarantined,
    mobility,
    rng
):
    new_positions = positions.copy()

    movable = (
        (states != DEAD)
        & (~hospitalized)
        & (~quarantined)
    )

    movement = rng.normal(
        loc=0.0,
        scale=BASE_MOVEMENT,
        size=(POPULATION, 2)
    )

    movement *= mobility[:, None]
    new_positions[movable] += movement[movable]
    new_positions[movable] = np.clip(
        new_positions[movable],
        0,
        CITY_SIZE
    )

    return new_positions


def transmission_probability(distance, vaccinated=False):
    probability = (
        TRANSMISSION_PROBABILITY
        * np.exp(
            -(distance ** 2)
            / (2 * DISTANCE_SCALE ** 2)
        )
    )

    if vaccinated:
        probability *= (1.0 - VACCINE_EFFECTIVENESS)

    return float(np.clip(probability, 0.0, 1.0))


def hospitalization_probability(age):
    risk = (
        0.03
        + 0.0015 * np.maximum(age - 40, 0)
    )
    return float(np.clip(risk, 0.03, 0.30))


def mortality_probability(
    age,
    requires_hospital,
    was_hospitalized
):
    base = 0.003
    age_component = (
        0.00015 * np.maximum(age - 40, 0)
    )

    if requires_hospital and not was_hospitalized:
        severity_component = 0.08
    elif requires_hospital and was_hospitalized:
        severity_component = 0.02
    else:
        severity_component = 0.0

    probability = (
        base
        + age_component
        + severity_component
    )

    return float(np.clip(probability, 0.0, 0.50))


def spread_infection(
    positions,
    states,
    vaccinated,
    quarantined,
    rng
):
    new_states = states.copy()

    infectious_indices = np.flatnonzero(
        (states == INFECTED) & (~quarantined)
    )

    susceptible_indices = np.flatnonzero(
        states == SUSCEPTIBLE
    )

    if len(infectious_indices) == 0 or len(susceptible_indices) == 0:
        return new_states, np.array([], dtype=np.int64), 0

    susceptible_positions = positions[susceptible_indices]
    tree = cKDTree(susceptible_positions)
    infection_risk = np.zeros(
        len(susceptible_indices),
        dtype=np.float64
    )

    for infected_idx in infectious_indices:
        infected_position = positions[infected_idx]
        nearby = tree.query_ball_point(
            infected_position,
            r=INFECTION_RADIUS
        )

        if not nearby:
            continue

        nearby_positions = susceptible_positions[nearby]
        distances = np.linalg.norm(
            nearby_positions - infected_position,
            axis=1
        )

        for local_idx, distance in zip(nearby, distances):
            person_idx = susceptible_indices[local_idx]
            probability = transmission_probability(
                distance,
                bool(vaccinated[person_idx])
            )

            infection_risk[local_idx] = (
                1.0
                - (1.0 - infection_risk[local_idx])
                * (1.0 - probability)
            )

    newly_exposed_mask = (
        rng.random(len(infection_risk))
        < infection_risk
    )

    newly_exposed_indices = susceptible_indices[
        newly_exposed_mask
    ]

    new_states[newly_exposed_indices] = EXPOSED

    return (
        new_states,
        newly_exposed_indices,
        int(len(newly_exposed_indices))
    )

In [ ]:
#CITY INFURSTRUCTURE FACILITES

def create_initial_hospitals():
    return pd.DataFrame({
        "id": np.arange(2),
        "x": [20.0, 80.0],
        "y": [80.0, 20.0],
        "capacity": [40, 60],
        "occupied": [0, 0]
    })


def create_initial_shelters():
    return pd.DataFrame({
        "id": np.arange(3),
        "x": [15.0, 75.0, 50.0],
        "y": [20.0, 70.0, 85.0],
        "capacity": [50, 60, 40],
        "occupied": [0, 0, 0],
        "open": [False, False, False]
    })


def create_initial_ambulances():
    return pd.DataFrame({
        "id": np.arange(NUM_AMBULANCES),
        "x": np.full(NUM_AMBULANCES, CITY_SIZE / 2),
        "y": np.full(NUM_AMBULANCES, CITY_SIZE / 2),
        "available": [True] * NUM_AMBULANCES,
        "target_person": [-1] * NUM_AMBULANCES,
        "target_hospital": [-1] * NUM_AMBULANCES,
        "busy_until": [0] * NUM_AMBULANCES
    })


def create_initial_resources():
    return {
        "vaccines": int(INITIAL_VACCINES),
        "tests": int(INITIAL_TESTS),
        "budget": int(INITIAL_BUDGET)
    }

In [ ]:
#City State and Action Helpers
def get_zone_statistics(
    zone_id,
    person_zones,
    states,
    vaccinated,
    quarantined,
    hospitalized
):
    mask = person_zones == zone_id
    population = int(np.sum(mask))

    return {
        "zone": int(zone_id),
        "population": population,
        "susceptible": int(np.sum(mask & (states == SUSCEPTIBLE))),
        "exposed": int(np.sum(mask & (states == EXPOSED))),
        "infected": int(np.sum(mask & (states == INFECTED))),
        "recovered": int(np.sum(mask & (states == RECOVERED))),
        "dead": int(np.sum(mask & (states == DEAD))),
        "vaccinated": int(np.sum(mask & vaccinated)),
        "quarantined": int(np.sum(mask & quarantined)),
        "hospitalized": int(np.sum(mask & hospitalized))
    }


def make_city_state(
    person_zones,
    states,
    vaccinated,
    quarantined,
    hospitalized
):
    rows = [
        get_zone_statistics(
            zone_id,
            person_zones,
            states,
            vaccinated,
            quarantined,
            hospitalized
        )
        for zone_id in range(NUM_ZONES)
    ]
    return pd.DataFrame(rows)


def get_action_description(env, action):
    decoded = env.decode_action(int(action))
    action_type = decoded["type"]

    if action_type == NO_ACTION:
        return "Do Nothing"
    if action_type == VACCINATE:
        return f"Vaccinate Zone {decoded['zone']} ({decoded['amount']} people)"
    if action_type == OPEN_SHELTER:
        return f"Open Shelter {decoded['shelter_id']}"
    if action_type == QUARANTINE:
        return f"Quarantine Zone {decoded['zone']} with Shelter {decoded['shelter_id']}"
    if action_type == DISPATCH_AMBULANCE:
        return f"Dispatch Ambulance to Zone {decoded['zone']}"
    return "Unknown Action"

## V5.5 environment and reward function

The cells below preserve the original environment code so the dashboard uses the same observation and transition logic as training.

In [ ]:
#REWARD FUNCTION

REWARD_WEIGHTS = {
    "new_exposures": 1.0,
    "deaths": 25.0,
    "hospital_overflow": 15.0,
    "action_cost": 0.01,
    "failed_action": 2.0,
    "successful_action": 0.05
}


def calculate_reward(metrics, action_result):
    reward = 0.0

    reward -= (
        REWARD_WEIGHTS["new_exposures"]
        * metrics["new_exposures"]
    )

    reward -= (
        REWARD_WEIGHTS["deaths"]
        * metrics["deaths_today"]
    )

    reward -= (
        REWARD_WEIGHTS["hospital_overflow"]
        * metrics["hospital_overflow"]
    )

    reward -= (
        REWARD_WEIGHTS["action_cost"]
        * action_result.get("cost", 0)
    )

    if action_result.get("success", False):
        reward += REWARD_WEIGHTS["successful_action"]
    else:
        reward -= REWARD_WEIGHTS["failed_action"]

    return float(reward)

In [ ]:
# AI ENV

class ImprovedEpiControlEnv(gym.Env):

    metadata = {"render_modes": []}

    def __init__(self, max_days=SIMULATION_DAYS):
        super().__init__()

        self.max_days = int(max_days)

        # Fixed initial population for reproducibility.
        self.initial_positions = INITIAL_POSITIONS.copy()
        self.initial_ages = INITIAL_AGES.copy()
        self.initial_mobility = INITIAL_MOBILITY.copy()
        self.initial_states = INITIAL_STATES.copy()
        self.initial_exposure_day = INITIAL_EXPOSURE_DAY.copy()
        self.initial_needs_hospital = INITIAL_NEEDS_HOSPITAL.copy()
        self.initial_hospitalized = INITIAL_HOSPITALIZED.copy()


        #Action space
        self.vaccine_amounts = np.array([10, 25, 50], dtype=np.int64)
        self.num_shelters = len(create_initial_shelters())
        self.num_vaccine_actions = NUM_ZONES * len(self.vaccine_amounts)
        self.num_quarantine_actions = NUM_ZONES * self.num_shelters
        self.num_ambulance_actions = NUM_ZONES

        self.none_action = 0
        self.vaccine_start = 1
        self.shelter_start = self.vaccine_start + self.num_vaccine_actions
        self.quarantine_start = self.shelter_start + self.num_shelters
        self.ambulance_start = self.quarantine_start + self.num_quarantine_actions
        self.total_actions = self.ambulance_start + self.num_ambulance_actions

        self.action_space = spaces.Discrete(self.total_actions)


        # Observation space
        # 12 features per zone + 8 global
        self.zone_features = 12
        self.global_features = 8
        self.observation_size = (
            NUM_ZONES * self.zone_features
            + self.global_features
        )

        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.observation_size,),
            dtype=np.float32
        )

        self._reset_state()

    def _reset_state(self):
        self.positions = self.initial_positions.copy()
        self.ages = self.initial_ages.copy()
        self.mobility = self.initial_mobility.copy()
        self.states = self.initial_states.copy()
        self.exposure_day = self.initial_exposure_day.copy()
        self.needs_hospital = self.initial_needs_hospital.copy()
        self.hospitalized = self.initial_hospitalized.copy()

        self.vaccinated = np.zeros(POPULATION, dtype=bool)
        self.quarantined = np.zeros(POPULATION, dtype=bool)
        self.person_zones = update_person_zones(self.positions)

        self.hospitals = create_initial_hospitals()
        self.shelters = create_initial_shelters()
        self.ambulances = create_initial_ambulances()
        self.resources = create_initial_resources()

        self.current_day = 0
        self.previous_zone_state = None


    #Gym reset
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._reset_state()

        observation = self._get_observation(
            reference_state=None
        )

        self.previous_zone_state = self._get_city_state().copy()
        info = self._get_info()
        return observation, info


    #Action decoding
    def decode_action(self, action):
        action = int(action)

        if action == self.none_action:
            return {"type": NO_ACTION}

        if self.vaccine_start <= action < self.shelter_start:
            offset = action - self.vaccine_start
            zone = offset // len(self.vaccine_amounts)
            amount_index = offset % len(self.vaccine_amounts)
            return {
                "type": VACCINATE,
                "zone": int(zone),
                "amount": int(self.vaccine_amounts[amount_index])
            }

        if self.shelter_start <= action < self.quarantine_start:
            return {
                "type": OPEN_SHELTER,
                "shelter_id": int(action - self.shelter_start)
            }

        if self.quarantine_start <= action < self.ambulance_start:
            offset = action - self.quarantine_start
            zone = offset // self.num_shelters
            shelter_id = offset % self.num_shelters
            return {
                "type": QUARANTINE,
                "zone": int(zone),
                "shelter_id": int(shelter_id)
            }

        if self.ambulance_start <= action < self.total_actions:
            return {
                "type": DISPATCH_AMBULANCE,
                "zone": int(action - self.ambulance_start)
            }

        raise ValueError(f"Unknown action: {action}")


    # City state
    def _get_city_state(self):
        return make_city_state(
            self.person_zones,
            self.states,
            self.vaccinated,
            self.quarantined,
            self.hospitalized
        )


    # Observation
    def _get_observation(self, reference_state=None):
        city_state = self._get_city_state()
        values = []

        for _, zone in city_state.iterrows():
            population = max(int(zone["population"]), 1)

            infection_trend = 0.0
            exposure_trend = 0.0
            hospitalization_trend = 0.0

            if reference_state is not None:
                previous = reference_state[
                    reference_state["zone"] == zone["zone"]
                ]

                if not previous.empty:
                    previous = previous.iloc[0]
                    infection_trend = (
                        zone["infected"] - previous["infected"]
                    ) / population
                    exposure_trend = (
                        zone["exposed"] - previous["exposed"]
                    ) / population
                    hospitalization_trend = (
                        zone["hospitalized"] - previous["hospitalized"]
                    ) / max(HOSPITAL_CAPACITY, 1)

            infection_trend = (np.clip(infection_trend, -1.0, 1.0) + 1.0) / 2.0
            exposure_trend = (np.clip(exposure_trend, -1.0, 1.0) + 1.0) / 2.0
            hospitalization_trend = (np.clip(hospitalization_trend, -1.0, 1.0) + 1.0) / 2.0

            values.extend([
                zone["population"] / POPULATION,
                zone["susceptible"] / POPULATION,
                zone["exposed"] / POPULATION,
                zone["infected"] / POPULATION,
                zone["recovered"] / POPULATION,
                zone["dead"] / POPULATION,
                zone["vaccinated"] / POPULATION,
                zone["quarantined"] / POPULATION,
                min(zone["hospitalized"] / HOSPITAL_CAPACITY, 1.0),
                infection_trend,
                exposure_trend,
                hospitalization_trend
            ])

        open_shelters = int(self.shelters["open"].sum())
        available_ambulances = int(self.ambulances["available"].sum())
        hospital_occupancy = int(np.sum(self.hospitalized))

        values.extend([
            self.resources["vaccines"] / INITIAL_VACCINES,
            self.resources["tests"] / INITIAL_TESTS,
            self.resources["budget"] / INITIAL_BUDGET,
            min(hospital_occupancy / HOSPITAL_CAPACITY, 1.0),
            open_shelters / max(self.num_shelters, 1),
            available_ambulances / max(NUM_AMBULANCES, 1),
            self.current_day / max(self.max_days, 1),
            np.sum(self.states == INFECTED) / POPULATION
        ])

        observation = np.asarray(values, dtype=np.float32)
        return np.clip(observation, 0.0, 1.0)


    # Intervention helpers
    def _vaccinate_zone(self, zone, amount):
        if self.resources["vaccines"] <= 0:
            return 0

        candidates = np.flatnonzero(
            (self.person_zones == zone)
            & (self.states == SUSCEPTIBLE)
            & (~self.vaccinated)
        )

        count = min(len(candidates), int(amount), self.resources["vaccines"])

        if count <= 0:
            return 0

        selected = candidates[:count]
        self.vaccinated[selected] = True
        self.resources["vaccines"] -= count
        return int(count)

    def _open_shelter(self, shelter_id):
        if self.shelters.loc[shelter_id, "open"]:
            return False

        if self.resources["budget"] < RESOURCE_COSTS["open_shelter"]:
            return False

        self.shelters.loc[shelter_id, "open"] = True
        self.resources["budget"] -= RESOURCE_COSTS["open_shelter"]
        return True

    def _quarantine_zone(self, zone, shelter_id):
        if not bool(self.shelters.loc[shelter_id, "open"]):
            return 0

        capacity_left = (
            self.shelters.loc[shelter_id, "capacity"]
            - self.shelters.loc[shelter_id, "occupied"]
        )

        if capacity_left <= 0:
            return 0

        candidates = np.flatnonzero(
            (self.person_zones == zone)
            & ((self.states == EXPOSED) | (self.states == INFECTED))
            & (~self.quarantined)
        )

        budget_limit = (
            self.resources["budget"]
            // RESOURCE_COSTS["quarantine_person"]
        )

        count = min(
            len(candidates),
            int(capacity_left),
            int(budget_limit)
        )

        if count <= 0:
            return 0

        selected = candidates[:count]
        self.quarantined[selected] = True
        self.shelters.loc[shelter_id, "occupied"] += count
        self.resources["budget"] -= count * RESOURCE_COSTS["quarantine_person"]
        return int(count)

    def _dispatch_ambulance(self, zone):
        available = self.ambulances[
            self.ambulances["available"]
        ]

        if available.empty:
            return None

        if self.resources["budget"] < RESOURCE_COSTS["ambulance_dispatch"]:
            return None

        candidates = np.flatnonzero(
            (self.person_zones == zone)
            & (self.states == INFECTED)
            & self.needs_hospital
            & (~self.hospitalized)
        )

        if len(candidates) == 0:
            return None

        patient = candidates[np.argmax(self.ages[candidates])]
        patient_position = self.positions[patient]

        available_hospitals = self.hospitals[
            self.hospitals["occupied"] < self.hospitals["capacity"]
        ]

        if available_hospitals.empty:
            return None

        distances = np.sqrt(
            (available_hospitals["x"].values - patient_position[0]) ** 2
            + (available_hospitals["y"].values - patient_position[1]) ** 2
        )

        hospital_id = int(
            available_hospitals.iloc[np.argmin(distances)]["id"]
        )

        ambulance_id = int(available.iloc[0]["id"])

        self.ambulances.loc[
            self.ambulances["id"] == ambulance_id,
            "available"
        ] = False

        self.ambulances.loc[
            self.ambulances["id"] == ambulance_id,
            "target_person"
        ] = patient

        self.ambulances.loc[
            self.ambulances["id"] == ambulance_id,
            "target_hospital"
        ] = hospital_id

        self.ambulances.loc[
            self.ambulances["id"] == ambulance_id,
            "busy_until"
        ] = self.current_day + 1

        self.hospitalized[patient] = True

        self.hospitals.loc[
            self.hospitals["id"] == hospital_id,
            "occupied"
        ] += 1

        self.resources["budget"] -= RESOURCE_COSTS["ambulance_dispatch"]

        return {
            "ambulance_id": ambulance_id,
            "patient_id": int(patient),
            "hospital_id": hospital_id
        }


    #Apply action

    def _apply_action(self, action):
        action_type = action["type"]

        if action_type == NO_ACTION:
            return {
                "type": NO_ACTION,
                "success": True,
                "cost": 0
            }

        if action_type == VACCINATE:
            count = self._vaccinate_zone(
                action["zone"],
                action["amount"]
            )
            return {
                "type": VACCINATE,
                "zone": action["zone"],
                "amount_delivered": count,
                "success": count > 0,
                "cost": count * RESOURCE_COSTS["vaccination"]
            }

        if action_type == OPEN_SHELTER:
            success = self._open_shelter(
                action["shelter_id"]
            )
            return {
                "type": OPEN_SHELTER,
                "shelter_id": action["shelter_id"],
                "success": success,
                "cost": RESOURCE_COSTS["open_shelter"] if success else 0
            }

        if action_type == QUARANTINE:
            count = self._quarantine_zone(
                action["zone"],
                action["shelter_id"]
            )
            return {
                "type": QUARANTINE,
                "zone": action["zone"],
                "shelter_id": action["shelter_id"],
                "quarantined": count,
                "success": count > 0,
                "cost": count * RESOURCE_COSTS["quarantine_person"]
            }

        if action_type == DISPATCH_AMBULANCE:
            result = self._dispatch_ambulance(
                action["zone"]
            )
            return {
                "type": DISPATCH_AMBULANCE,
                "zone": action["zone"],
                "success": result is not None,
                "details": result,
                "cost": RESOURCE_COSTS["ambulance_dispatch"] if result else 0
            }

        raise ValueError(f"Unsupported action type: {action_type}")


    #1-day sim

    def _simulate_one_day(self):
        # Release ambulances whose one-day service period is over.
        for idx in self.ambulances.index:
            if (
                not bool(self.ambulances.loc[idx, "available"])
                and self.ambulances.loc[idx, "busy_until"] <= self.current_day
            ):
                self.ambulances.loc[idx, "available"] = True
                self.ambulances.loc[idx, "target_person"] = -1
                self.ambulances.loc[idx, "target_hospital"] = -1

        self.positions = move_population(
            self.positions,
            self.states,
            self.hospitalized,
            self.quarantined,
            self.mobility,
            self.np_random
        )

        self.person_zones = update_person_zones(
            self.positions
        )

        (
            self.states,
            newly_exposed_indices,
            new_exposures
        ) = spread_infection(
            self.positions,
            self.states,
            self.vaccinated,
            self.quarantined,
            self.np_random
        )

        for idx in newly_exposed_indices:
            self.exposure_day[idx] = self.current_day

        exposed_indices = np.flatnonzero(
            self.states == EXPOSED
        )

        for idx in exposed_indices:
            if (
                self.current_day - self.exposure_day[idx]
                >= INCUBATION_DAYS
            ):
                self.states[idx] = INFECTED
                self.needs_hospital[idx] = (
                    self.np_random.random()
                    < hospitalization_probability(self.ages[idx])
                )

        current_occupancy = int(np.sum(self.hospitalized))
        available_beds = max(
            0,
            HOSPITAL_CAPACITY - current_occupancy
        )

        hospital_candidates = np.flatnonzero(
            (self.states == INFECTED)
            & self.needs_hospital
            & (~self.hospitalized)
        )

        if available_beds > 0 and len(hospital_candidates) > 0:
            priority = hospital_candidates[
                np.argsort(self.ages[hospital_candidates])[::-1]
            ]
            selected = priority[:available_beds]
            self.hospitalized[selected] = True

            remaining_candidates = set(hospital_candidates.tolist()) - set(selected.tolist())
            for idx in remaining_candidates:
                pass

        deaths_today = 0
        recoveries_today = 0

        infected_indices = np.flatnonzero(
            self.states == INFECTED
        )

        for idx in infected_indices:
            disease_duration = (
                self.current_day - self.exposure_day[idx]
            )

            if disease_duration >= INCUBATION_DAYS + INFECTIOUS_DAYS:
                dies = (
                    self.np_random.random()
                    < mortality_probability(
                        self.ages[idx],
                        bool(self.needs_hospital[idx]),
                        bool(self.hospitalized[idx])
                    )
                )

                if dies:
                    self.states[idx] = DEAD
                    self.hospitalized[idx] = False
                    self.quarantined[idx] = False
                    deaths_today += 1
                else:
                    self.states[idx] = RECOVERED
                    self.hospitalized[idx] = False
                    self.quarantined[idx] = False
                    recoveries_today += 1

        return {
            "new_exposures": int(new_exposures),
            "deaths_today": int(deaths_today),
            "recoveries_today": int(recoveries_today)
        }

    #metrics
    def _get_metrics(self, daily):
        hospitalized_count = int(np.sum(self.hospitalized))
        return {
            "day": int(self.current_day),
            "population": POPULATION,
            "susceptible": int(np.sum(self.states == SUSCEPTIBLE)),
            "exposed": int(np.sum(self.states == EXPOSED)),
            "infected": int(np.sum(self.states == INFECTED)),
            "recovered": int(np.sum(self.states == RECOVERED)),
            "dead": int(np.sum(self.states == DEAD)),
            "hospitalized": hospitalized_count,
            "vaccinated": int(np.sum(self.vaccinated)),
            "quarantined": int(np.sum(self.quarantined)),
            "new_exposures": int(daily["new_exposures"]),
            "deaths_today": int(daily["deaths_today"]),
            "recoveries_today": int(daily["recoveries_today"]),
            "hospital_overflow": max(0, hospitalized_count - HOSPITAL_CAPACITY)
        }

    def _get_info(self):
        return {
            "day": int(self.current_day),
            "population": POPULATION,
            "infected": int(np.sum(self.states == INFECTED)),
            "resources": self.resources.copy()
        }


    #Step
    def step(self, action):
        reference_state = self._get_city_state().copy()

        decoded_action = self.decode_action(action)
        action_result = self._apply_action(decoded_action)

        self.current_day += 1
        daily = self._simulate_one_day()
        metrics = self._get_metrics(daily)

        reward = calculate_reward(
            metrics,
            action_result
        )

        active_cases = int(
            np.sum(self.states == EXPOSED)
            + np.sum(self.states == INFECTED)
        )

        terminated = bool(active_cases == 0)
        truncated = bool(self.current_day >= self.max_days)

        observation = self._get_observation(
            reference_state=reference_state
        )

        self.previous_zone_state = self._get_city_state().copy()

        info = {
            "action": decoded_action,
            "action_result": action_result,
            "metrics": metrics,
            "resources": self.resources.copy()
        }

        return (
            observation,
            float(reward),
            terminated,
            truncated,
            info
        )

    def render(self):
        metrics = self._get_metrics({
            "new_exposures": 0,
            "deaths_today": 0,
            "recoveries_today": 0
        })
        print(
            f"Day {metrics['day']} | "
            f"Infected: {metrics['infected']} | "
            f"Dead: {metrics['dead']} | "
            f"Hospitalized: {metrics['hospitalized']}"
        )

## V5.6 strategic action space

Actions are: **0** no intervention, **1** vaccinate the best zone, **2** open a shelter, **3** quarantine the best zone, and **4** dispatch an ambulance.

In [ ]:
STRATEGIC_NO_ACTION = 0
STRATEGIC_VACCINATE = 1
STRATEGIC_OPEN_SHELTER = 2
STRATEGIC_QUARANTINE = 3
STRATEGIC_AMBULANCE = 4

class StrategicEpiControlEnv(ImprovedEpiControlEnv):
    def __init__(self, max_days=SIMULATION_DAYS):
        super().__init__(max_days=max_days)
        self.action_space = spaces.Discrete(5)
        self.total_actions = 5

    def _zone_trend(self, zone_id, column):
        if self.previous_zone_state is None:
            return 0.0
        current = self._get_city_state()
        current_row = current[current["zone"] == zone_id]
        previous_row = self.previous_zone_state[self.previous_zone_state["zone"] == zone_id]
        if current_row.empty or previous_row.empty:
            return 0.0
        return float(current_row.iloc[0][column] - previous_row.iloc[0][column])

    def _best_vaccine_zone(self):
        city = self._get_city_state()
        scores = []
        for _, row in city.iterrows():
            zone = int(row["zone"])
            population = max(int(row["population"]), 1)
            unvaccinated_susceptible = max(int(row["susceptible"]) - int(row["vaccinated"]), 0)
            if unvaccinated_susceptible <= 0:
                scores.append((-1.0, zone))
                continue
            infection_pressure = (
                4.0 * int(row["infected"])
                + 2.0 * int(row["exposed"])
                + 0.5 * int(row["hospitalized"])
            )
            trend_pressure = (
                3.0 * max(self._zone_trend(zone, "infected"), 0.0)
                + 1.5 * max(self._zone_trend(zone, "exposed"), 0.0)
            )
            local_pressure = (infection_pressure + trend_pressure) / population
            score = unvaccinated_susceptible * (1.0 + local_pressure)
            scores.append((score, zone))
        return int(max(scores, key=lambda item: item[0])[1])

    def _best_quarantine_zone(self):
        city = self._get_city_state()
        scores = []
        for _, row in city.iterrows():
            zone = int(row["zone"])
            active = int(row["infected"]) + int(row["exposed"])
            if active <= 0:
                scores.append((-1.0, zone))
                continue
            population = max(int(row["population"]), 1)
            trend_pressure = (
                3.0 * max(self._zone_trend(zone, "infected"), 0.0)
                + 2.0 * max(self._zone_trend(zone, "exposed"), 0.0)
            )
            score = (
                4.0 * int(row["infected"])
                + 2.0 * int(row["exposed"])
                + 0.5 * int(row["hospitalized"])
                + trend_pressure
            ) / population
            score *= active
            scores.append((score, zone))
        return int(max(scores, key=lambda item: item[0])[1])

    def _best_ambulance_zone(self):
        zone_scores = np.zeros(NUM_ZONES, dtype=np.float64)
        candidates = np.flatnonzero(
            (self.states == INFECTED)
            & self.needs_hospital
            & (~self.hospitalized)
        )
        for person_idx in candidates:
            zone = int(self.person_zones[person_idx])
            age_factor = 1.0 + max(0.0, float(self.ages[person_idx]) - 40.0) / 50.0
            zone_scores[zone] += age_factor
        if np.max(zone_scores) > 0:
            return int(np.argmax(zone_scores))
        return self._best_quarantine_zone()

    def _best_open_shelter(self):
        closed = self.shelters[~self.shelters["open"]]
        if closed.empty:
            return 0
        closed = closed.copy()
        closed["remaining_capacity"] = closed["capacity"] - closed["occupied"]
        return int(closed.sort_values(["remaining_capacity", "capacity"], ascending=False).iloc[0]["id"])

    def _best_open_shelter_for_quarantine(self):
        open_shelters = self.shelters[self.shelters["open"]].copy()
        if open_shelters.empty:
            return None
        open_shelters["remaining_capacity"] = open_shelters["capacity"] - open_shelters["occupied"]
        open_shelters = open_shelters[open_shelters["remaining_capacity"] > 0]
        if open_shelters.empty:
            return None
        return int(open_shelters.sort_values("remaining_capacity", ascending=False).iloc[0]["id"])

    def decode_action(self, action):
        action = int(action)
        if action == STRATEGIC_NO_ACTION:
            return {"type": NO_ACTION, "strategic_action": STRATEGIC_NO_ACTION}
        if action == STRATEGIC_VACCINATE:
            return {"type": VACCINATE, "zone": self._best_vaccine_zone(), "amount": 25, "strategic_action": STRATEGIC_VACCINATE}
        if action == STRATEGIC_OPEN_SHELTER:
            return {"type": OPEN_SHELTER, "shelter_id": self._best_open_shelter(), "strategic_action": STRATEGIC_OPEN_SHELTER}
        if action == STRATEGIC_QUARANTINE:
            shelter_id = self._best_open_shelter_for_quarantine()
            if shelter_id is None:
                shelter_id = self._best_open_shelter()
            return {"type": QUARANTINE, "zone": self._best_quarantine_zone(), "shelter_id": shelter_id, "strategic_action": STRATEGIC_QUARANTINE}
        if action == STRATEGIC_AMBULANCE:
            return {"type": DISPATCH_AMBULANCE, "zone": self._best_ambulance_zone(), "strategic_action": STRATEGIC_AMBULANCE}
        raise ValueError(f"Unknown strategic action: {action}")


def strategic_action_description(env, action):
    decoded = env.decode_action(int(action))
    if decoded["type"] == NO_ACTION:
        return "Do Nothing"
    if decoded["type"] == VACCINATE:
        return f"Vaccinate Best Zone -> Zone {decoded['zone']} (25 people)"
    if decoded["type"] == OPEN_SHELTER:
        return f"Open Best Shelter -> Shelter {decoded['shelter_id']}"
    if decoded["type"] == QUARANTINE:
        return f"Quarantine Best Zone -> Zone {decoded['zone']} using Shelter {decoded['shelter_id']}"
    if decoded["type"] == DISPATCH_AMBULANCE:
        return f"Dispatch Ambulance -> Zone {decoded['zone']}"
    return "Unknown Action"

In [ ]:
# Load the existing trained V5.6 checkpoint — NO TRAINING.
from pathlib import Path

V56_ROOT = Path('/content/drive/MyDrive/EpiControl/DQN_V56_Strategic')
V56_BEST_PATH = V56_ROOT / 'best_model' / 'best_model.zip'
V56_FINAL_PATH = V56_ROOT / 'final' / 'dqn_v56_strategic_final.zip'

if V56_BEST_PATH.exists():
    V56_MODEL_PATH = V56_BEST_PATH
elif V56_FINAL_PATH.exists():
    V56_MODEL_PATH = V56_FINAL_PATH
else:
    raise FileNotFoundError(
        'Could not find the V5.6 model in Google Drive. Expected one of:\n'
        f'  {V56_BEST_PATH}\n  {V56_FINAL_PATH}'
    )

dqn_model = DQN.load(str(V56_MODEL_PATH), device='auto')
probe_env = StrategicEpiControlEnv(max_days=SIMULATION_DAYS)
probe_observation, _ = probe_env.reset(seed=42)

if int(getattr(dqn_model.action_space, 'n', -1)) != 5:
    raise ValueError(f'Expected a 5-action V5.6 model, found action space: {dqn_model.action_space}')
if tuple(dqn_model.observation_space.shape or ()) != tuple(probe_observation.shape):
    raise ValueError(
        f'Model/environment observation mismatch: model={dqn_model.observation_space.shape}, '
        f'environment={probe_observation.shape}'
    )
_probe_action, _ = dqn_model.predict(probe_observation, deterministic=True)
print('Loaded checkpoint:', V56_MODEL_PATH)
print('Observation shape:', probe_observation.shape, '| action space:', dqn_model.action_space)
print('Prediction smoke test passed; action:', int(np.asarray(_probe_action).reshape(-1)[0]))

Loaded checkpoint: /content/drive/MyDrive/EpiControl/DQN_V56_Strategic/best_model/best_model.zip
Observation shape: (308,) | action space: Discrete(5)
Prediction smoke test passed; action: 2


In [ ]:
%pip -q install fastapi uvicorn shapely requests

In [ ]:
from pathlib import Path
WEB_ROOT = Path('/content/epicontrol_web')
WEB_ROOT.mkdir(parents=True, exist_ok=True)
print('Web asset folder ready:', WEB_ROOT)

Web asset folder ready: /content/epicontrol_web


In [ ]:
%%writefile /content/epicontrol_web/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <meta name="theme-color" content="#08111c" />
  <title>EpiControl | Monaco Response Console</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=DM+Mono:wght@400;500&family=Manrope:wght@400;500;600;700;800&display=swap" rel="stylesheet">
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" crossorigin="" />
  <link rel="stylesheet" href="/static/style.css" />
</head>
<body>
  <div class="ambient ambient-one"></div><div class="ambient ambient-two"></div>
  <header class="topbar">
    <div class="brand-lockup">
      <div class="brand-mark" aria-hidden="true"><span></span><span></span><span></span><span></span></div>
      <div class="brand-copy"><div class="eyebrow">AUTONOMOUS EPIDEMIC RESPONSE</div><div class="brand-title">EpiControl <span>MONACO / 01</span></div></div>
    </div>
    <div class="header-status">
      <div class="connection-pill"><span class="pulse-dot"></span><span id="connection-label">CONNECTING</span></div>
      <div class="day-counter"><span class="eyebrow">SIMULATION DAY</span><strong id="day-number">00 <small>/ 60</small></strong></div>
    </div>
  </header>

  <main class="page-shell">
    <section class="intro-row">
      <div class="intro-copy">
        <div class="eyebrow accent">DIGITAL TWIN <span class="slash">/</span> LIVE EPIDEMIC MODEL</div>
        <h1>See the spread.<br><em>Move the response.</em></h1>
        <p>A live, synthetic epidemic scenario across Monaco. First watch an uncontrolled outbreak spread across the city, then hand the same active outbreak to the trained DQN agent.</p>
      </div>
      <div class="controls-wrap">
        <div class="run-controls">
          <button class="btn btn-primary" id="start-baseline"><span class="button-symbol">▶</span><span><strong>Start pandemic</strong><small>Uncontrolled spread · no recoveries</small></span></button>
          <button class="btn btn-agent" id="start-agent"><span class="button-symbol">✳</span><span><strong>Start DQN agent</strong><small>Start pandemic phase first</small></span></button>
        </div>
        <div class="secondary-controls">
          <button class="btn btn-quiet" id="pause-btn" disabled>Ⅱ &nbsp; Pause</button>
          <button class="btn btn-quiet" id="reset-btn">↻ &nbsp; Reset</button>
          <label class="speed-control">SPEED <select id="speed-select" aria-label="Simulation speed"><option value="0.75">0.75×</option><option value="1" selected>1×</option><option value="1.5">1.5×</option><option value="2">2×</option></select></label>
        </div>
      </div>
    </section>

    <section class="metric-grid" aria-label="Live health metrics">
      <article class="metric-card metric-pop"><div class="metric-label">POPULATION</div><div class="metric-value" id="m-population">1,000</div><div class="metric-foot">Synthetic population</div></article>
      <article class="metric-card metric-infected"><div class="metric-label"><span class="status-dot dot-red"></span> INFECTED</div><div class="metric-value" id="m-infected">5</div><div class="metric-foot" id="m-new-cases">0 new today</div></article>
      <article class="metric-card metric-exposed"><div class="metric-label"><span class="status-dot dot-amber"></span> EXPOSED</div><div class="metric-value" id="m-exposed">0</div><div class="metric-foot">Incubation phase</div></article>
      <article class="metric-card metric-recovered"><div class="metric-label"><span class="status-dot dot-blue"></span> RECOVERED</div><div class="metric-value" id="m-recovered">0</div><div class="metric-foot">Total recoveries</div></article>
      <article class="metric-card metric-dead"><div class="metric-label"><span class="status-dot dot-muted"></span> DECEASED</div><div class="metric-value" id="m-dead">0</div><div class="metric-foot">Total deaths</div></article>
      <article class="metric-card metric-vaccinated"><div class="metric-label"><span class="status-dot dot-green"></span> VACCINATED</div><div class="metric-value" id="m-vaccinated">0</div><div class="metric-foot">People protected</div></article>
    </section>

    <section class="workspace">
      <article class="map-card panel">
        <div class="panel-heading map-heading">
          <div><div class="panel-kicker"><span class="status-dot dot-cyan"></span> GEOSPATIAL MONITOR</div><h2>Monaco · live situation</h2></div>
          <div class="map-heading-right"><span class="mode-pill" id="mode-pill">STANDBY</span><span class="map-coordinates">43°44′ N&nbsp; / &nbsp;7°25′ E</span></div>
        </div>
        <div id="map-wrap">
          <div id="map" aria-label="Interactive Monaco epidemic simulation map"></div>
          <div class="map-glass-top"><span class="legend-chip"><i class="swatch swatch-infection"></i> Active spread</span><span class="legend-chip"><i class="swatch swatch-person"></i> Population</span><span class="legend-chip">✚ Response facilities</span></div>
          <div class="map-status-bar"><span class="live-map-tag"><i></i> LIVE MAP</span><span id="map-status">LOADING MONACO</span></div>
          <div class="map-loading" id="map-loading"><span class="loader-ring"></span><span>Connecting to Monaco map…</span></div>
        </div>
        <div class="map-legend">
          <span><i class="legend-dot color-susceptible"></i> Susceptible</span>
          <span><i class="legend-dot color-exposed"></i> Exposed</span>
          <span><i class="legend-dot color-infected"></i> Infected</span>
          <span><i class="legend-dot color-recovered"></i> Recovered</span>
          <span><i class="legend-dot color-dead"></i> Deceased</span>
          <span><i class="legend-dot color-ambulance"></i> Ambulance</span>
        </div>
        <div class="map-note">The epidemic, people, vehicles, and facilities are simulated. Their positions are visualized within Monaco; this is not real-world health data.</div>
      </article>

      <aside class="side-stack">
        <section class="panel response-panel">
          <div class="panel-heading"><div><div class="panel-kicker">RESPONSE CAPACITY</div><h2>Resources & readiness</h2></div><span class="panel-icon">⌁</span></div>
          <div class="hospital-block"><div class="resource-line"><span class="resource-name"><span class="resource-icon hospital-icon">＋</span> Hospital occupancy</span><strong id="hospital-text">0 / 100</strong></div><div class="progress-track"><div class="progress-fill" id="hospital-bar"></div></div></div>
          <div class="resource-list">
            <div class="resource-row"><span class="resource-icon quarantine-icon">⌑</span><span class="resource-label">Quarantined</span><strong id="m-quarantined">0</strong></div>
            <div class="resource-row"><span class="resource-icon vaccine-icon">✚</span><span class="resource-label">Vaccine doses</span><strong id="m-vaccine-stock">150</strong></div>
            <div class="resource-row"><span class="resource-icon budget-icon">¤</span><span class="resource-label">Response budget</span><strong id="m-budget">10,000</strong></div>
          </div>
          <div class="ambulance-status"><div class="ambulance-mini">🚑</div><div><span>MEDICAL FLEET</span><strong id="m-ambulances">5 / 5 available</strong></div><span class="fleet-pulse"></span></div>
        </section>

        <section class="panel agent-panel">
          <div class="panel-heading"><div><div class="panel-kicker">STRATEGIC CONTROL</div><h2>Agent activity</h2></div><div class="ai-mark">✳</div></div>
          <div class="agent-state-line"><span class="status-dot dot-cyan" id="agent-dot"></span><span id="agent-mode-text">AGENT STANDING BY</span><span class="day-tag" id="action-day-tag">DAY 00</span></div>
          <div class="decision-card"><div class="decision-label">LATEST DECISION</div><div class="decision-title" id="decision-title">Awaiting simulation start</div><p id="decision-detail">Start a scenario to watch the model evolve, or let the trained DQN agent choose interventions.</p></div>
          <div class="decision-foot"><div><span>LAST REWARD</span><strong id="last-reward">—</strong></div><div><span>OUTCOME</span><strong id="action-result">STANDBY</strong></div></div>
          <div class="activity-title"><span>RECENT ACTIVITY</span><span id="activity-count">0 EVENTS</span></div>
          <div class="action-feed" id="action-feed"><div class="feed-empty">Actions taken by the agent will appear here.</div></div>
        </section>
      </aside>
    </section>

    <footer class="footer"><span>EPICONTROL <b>V5.6 STRATEGIC DQN</b></span><span>MAP TILES © OPENSTREETMAP CONTRIBUTORS · BOUNDARY: geoBoundaries</span><span id="footer-status">CONNECTING TO LOCAL MODEL</span></footer>
  </main>
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js" crossorigin=""></script>
  <script src="/static/app.js"></script>
</body>
</html>


Writing /content/epicontrol_web/index.html


In [ ]:
%%writefile /content/epicontrol_web/style.css
:root{
  --bg:#08111c;--panel:#0f1b29;--panel-raised:#122236;--border:rgba(155,184,211,.13);
  --text:#edf5ff;--muted:#879bb0;--muted-2:#637b92;--cyan:#5be7dc;--cyan-dim:#2daea9;
  --red:#ff6578;--amber:#ffc36f;--blue:#7abaff;--green:#4fe0b2;--mono:'DM Mono',monospace;--sans:'Manrope',system-ui,sans-serif;
}
*{box-sizing:border-box}
html,body{margin:0;min-height:100%;background:var(--bg);color:var(--text);font-family:var(--sans)}
body{background:radial-gradient(ellipse at 76% -18%,rgba(26,136,154,.17),transparent 44%),linear-gradient(145deg,#07101a,#091522 54%,#08111c);overflow-x:hidden}
button,select{font:inherit}.ambient{position:fixed;z-index:0;pointer-events:none;filter:blur(95px);opacity:.11;border-radius:50%}.ambient-one{width:310px;height:310px;background:#21c9c1;right:-130px;top:24vh}.ambient-two{width:270px;height:270px;background:#3464cd;left:-175px;bottom:7vh}
.topbar,.page-shell{position:relative;z-index:1}.topbar{height:78px;padding:0 clamp(18px,3.1vw,48px);display:flex;align-items:center;justify-content:space-between;border-bottom:1px solid var(--border);background:rgba(7,15,25,.82);backdrop-filter:blur(18px)}.brand-lockup{display:flex;align-items:center;gap:14px}.brand-mark{width:31px;height:31px;display:grid;grid-template-columns:1fr 1fr;gap:3px;transform:rotate(45deg);flex:0 0 auto}.brand-mark span{background:var(--cyan);border-radius:2px;box-shadow:0 0 13px rgba(91,231,220,.24)}.brand-mark span:nth-child(2),.brand-mark span:nth-child(3){opacity:.42}.eyebrow,.panel-kicker,.metric-label{font:500 9px var(--mono);letter-spacing:1.35px;color:var(--muted)}.brand-title{font-size:21px;font-weight:800;letter-spacing:-.8px;margin-top:4px}.brand-title span{font:10px var(--mono);color:var(--cyan);letter-spacing:1.25px;margin-left:8px}.header-status{display:flex;align-items:center;gap:26px}.connection-pill{display:flex;align-items:center;gap:9px;border:1px solid rgba(91,231,220,.2);background:rgba(91,231,220,.045);padding:9px 12px;border-radius:999px;font:9px var(--mono);letter-spacing:1px;color:var(--cyan)}.pulse-dot{height:7px;width:7px;border-radius:50%;background:var(--cyan);box-shadow:0 0 0 4px rgba(91,231,220,.08);animation:soft-pulse 1.8s ease-in-out infinite}.day-counter{display:flex;align-items:center;gap:12px}.day-counter strong{font:500 20px var(--mono);letter-spacing:-1px}.day-counter small{font-size:10px;color:var(--muted)}
.page-shell{max-width:1900px;margin:0 auto;padding:0 clamp(14px,2.6vw,42px) 20px}.intro-row{display:flex;align-items:center;justify-content:space-between;gap:28px;padding:26px 0 21px}.intro-copy{max-width:600px}.accent{color:var(--cyan)}.slash{color:#38556c;padding:0 6px}h1{margin:8px 0 8px;font-size:clamp(27px,3vw,39px);line-height:1.13;font-weight:700;letter-spacing:-1.7px}h1 em{font-style:normal;color:var(--cyan)}.intro-copy p{max-width:540px;margin:0;color:#91a4b9;font-size:12px;line-height:1.7}.controls-wrap{display:flex;flex-direction:column;align-items:flex-end;gap:9px}.run-controls,.secondary-controls{display:flex;align-items:center;justify-content:flex-end;gap:9px}.btn{display:flex;align-items:center;gap:10px;min-height:47px;padding:8px 15px;border-radius:9px;border:1px solid transparent;cursor:pointer;transition:transform .16s ease,border-color .16s ease,background .16s ease,opacity .16s ease;color:var(--text)}.btn:hover:not(:disabled){transform:translateY(-1px)}.btn:disabled{opacity:.45;cursor:not-allowed}.btn-primary{background:linear-gradient(125deg,#48cfc8,#3baea8);color:#052127;box-shadow:0 6px 24px rgba(47,204,193,.1)}.btn-agent{background:linear-gradient(135deg,#1c3046,#15283c);border-color:rgba(91,231,220,.24);color:#e4f7ff}.button-symbol{font-size:17px;flex:0 0 auto}.btn strong,.btn small{display:block;text-align:left}.btn strong{font-size:11px;font-weight:800}.btn small{font-size:9px;opacity:.72;margin-top:2px}.btn-quiet{min-height:30px;padding:5px 11px;background:rgba(255,255,255,.025);border-color:var(--border);color:#9fb1c5;font-size:10px;border-radius:7px}.speed-control{display:flex;align-items:center;gap:8px;font:9px var(--mono);letter-spacing:1px;color:#7f93a8;margin-left:3px}.speed-control select{padding:5px 7px;background:#101d2b;border:1px solid var(--border);border-radius:6px;color:#dce9f6;font:10px var(--mono);outline:none}
.metric-grid{display:grid;grid-template-columns:repeat(6,minmax(0,1fr));gap:10px;margin:0 0 14px}.metric-card{position:relative;min-width:0;padding:13px 15px 12px;border:1px solid var(--border);border-radius:9px;background:linear-gradient(150deg,rgba(18,32,47,.94),rgba(12,24,37,.96));overflow:hidden;box-shadow:0 10px 28px rgba(0,0,0,.07)}.metric-card:after{content:"";position:absolute;left:0;right:0;bottom:0;height:2px;background:var(--metric-color,var(--cyan));opacity:.65}.metric-label{display:flex;align-items:center;gap:7px;font-size:8px;letter-spacing:1.25px}.metric-value{margin:9px 0 4px;font:500 clamp(22px,2.1vw,29px) var(--mono);letter-spacing:-1px;color:#f4f8fd;transition:color .2s}.metric-foot{font-size:9px;color:#7e92a8;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.metric-pop{--metric-color:var(--cyan)}.metric-infected{--metric-color:var(--red)}.metric-exposed{--metric-color:var(--amber)}.metric-recovered{--metric-color:var(--blue)}.metric-dead{--metric-color:#8999ac}.metric-vaccinated{--metric-color:var(--green)}.status-dot{display:inline-block;width:6px;height:6px;flex:0 0 6px;border-radius:50%}.dot-cyan{background:var(--cyan);color:var(--cyan)}.dot-red{background:var(--red);color:var(--red)}.dot-amber{background:var(--amber);color:var(--amber)}.dot-blue{background:var(--blue);color:var(--blue)}.dot-muted{background:#8a98aa;color:#8a98aa}.dot-green{background:var(--green);color:var(--green)}
.workspace{display:grid;grid-template-columns:minmax(0,1.9fr) minmax(315px,.76fr);gap:14px;align-items:stretch}.panel{min-width:0;background:linear-gradient(150deg,rgba(16,29,44,.97),rgba(11,22,35,.98));border:1px solid var(--border);border-radius:11px;box-shadow:0 18px 45px rgba(0,0,0,.12);overflow:hidden}.panel-heading{display:flex;align-items:center;justify-content:space-between;gap:12px;padding:16px 18px 13px}.panel-heading h2{margin:5px 0 0;font-size:14px;font-weight:800;letter-spacing:-.35px}.panel-kicker{font-size:8px;letter-spacing:1.45px}.map-heading-right{display:flex;align-items:center;gap:13px}.mode-pill{background:rgba(91,231,220,.07);border:1px solid rgba(91,231,220,.18);padding:7px 9px;border-radius:6px;color:var(--cyan);font:9px var(--mono);letter-spacing:.8px}.map-coordinates{font:9px var(--mono);color:#72879d;white-space:nowrap}
#map-wrap{position:relative;height:clamp(550px,68vh,840px);min-height:500px;background:#0a1521;isolation:isolate}#map{width:100%;height:100%;background:#0a1521;z-index:1}.leaflet-tile-pane img{filter:invert(1) hue-rotate(180deg) brightness(.72) contrast(1.12) saturate(.55)}.leaflet-container{font-family:var(--sans)!important;background:#0a1521!important}.map-glass-top,.map-status-bar{position:absolute;z-index:700;left:13px;right:13px;display:flex;align-items:center;gap:8px;pointer-events:none}.map-glass-top{top:13px;flex-wrap:wrap}.legend-chip{display:inline-flex;align-items:center;gap:7px;background:rgba(7,16,27,.83);border:1px solid rgba(170,198,223,.16);color:#c3d3e3;border-radius:6px;padding:7px 9px;font:9px var(--mono);backdrop-filter:blur(8px)}.swatch{width:8px;height:8px;border-radius:2px;display:inline-block}.swatch-infection{background:var(--red);box-shadow:0 0 9px rgba(255,101,120,.5)}.swatch-person{background:var(--cyan)}.map-status-bar{bottom:12px;justify-content:space-between;font:8px var(--mono);letter-spacing:1.1px;color:#bac8d7}.live-map-tag{display:flex;align-items:center;gap:7px;background:rgba(7,16,27,.87);padding:7px 9px;border:1px solid var(--border);border-radius:6px;color:var(--cyan)}.live-map-tag i{width:6px;height:6px;background:var(--cyan);border-radius:50%;animation:soft-pulse 1.4s infinite}.map-status-bar>span:last-child{padding:7px 9px;border-radius:6px;background:rgba(7,16,27,.87);border:1px solid var(--border)}.map-loading{position:absolute;inset:0;z-index:600;display:flex;gap:10px;align-items:center;justify-content:center;background:rgba(7,16,27,.8);color:#b6c8d9;font:10px var(--mono);letter-spacing:.5px;transition:opacity .4s ease}.map-loading.is-hidden{opacity:0;pointer-events:none}.loader-ring{width:15px;height:15px;border:2px solid rgba(91,231,220,.18);border-top-color:var(--cyan);border-radius:50%;animation:spin .75s linear infinite}.map-legend{display:flex;flex-wrap:wrap;gap:15px 18px;padding:13px 17px;border-top:1px solid rgba(155,184,211,.09);font:9px var(--mono);color:#9eb0c3}.map-legend span{display:flex;align-items:center;gap:6px}.legend-dot{width:7px;height:7px;border-radius:50%;display:inline-block}.color-susceptible{background:#2dd4bf}.color-exposed{background:#ffc36f}.color-infected{background:#ff6578;box-shadow:0 0 7px rgba(255,101,120,.4)}.color-recovered{background:#7abaff}.color-dead{background:#8190a4}.color-ambulance{background:#f5fbff}.map-note{padding:0 17px 13px;color:#687f96;font-size:9px;line-height:1.55}
.side-stack{display:flex;flex-direction:column;gap:14px}.response-panel{flex:0 0 auto}.panel-icon,.ai-mark{color:var(--cyan);font-size:20px;opacity:.88}.hospital-block{padding:2px 17px 14px}.resource-line{display:flex;align-items:center;justify-content:space-between;gap:10px}.resource-name{display:flex;align-items:center;gap:9px;color:#bccbda;font-size:10px}.resource-line>strong{font:10px var(--mono);color:#f0f6fc}.resource-icon{display:inline-flex;align-items:center;justify-content:center;width:28px;height:28px;flex:0 0 28px;border:1px solid var(--border);border-radius:7px;background:rgba(255,255,255,.025);font-size:15px}.hospital-icon{color:var(--red)}.quarantine-icon{color:var(--amber)}.vaccine-icon{color:var(--green)}.budget-icon{color:var(--blue)}.progress-track{height:5px;background:#223144;border-radius:999px;overflow:hidden;margin-top:10px}.progress-fill{height:100%;width:0;border-radius:999px;background:linear-gradient(90deg,#ffb66d,#ff6578);box-shadow:0 0 13px rgba(255,101,120,.22);transition:width .5s ease}.resource-list{border-top:1px solid rgba(155,184,211,.09);padding:4px 17px 7px}.resource-row{display:flex;align-items:center;gap:10px;padding:8px 0}.resource-label{flex:1;color:#94a8bc;font-size:10px}.resource-row strong{font:10px var(--mono);color:#edf4fc}.ambulance-status{display:flex;align-items:center;gap:10px;margin:0 17px 16px;padding:10px 11px;border:1px solid rgba(91,231,220,.13);background:rgba(91,231,220,.035);border-radius:8px}.ambulance-mini{display:flex;align-items:center;justify-content:center;width:32px;height:32px;border-radius:8px;background:rgba(91,231,220,.08);font-size:19px}.ambulance-status>div:nth-child(2){display:flex;flex-direction:column;gap:4px;flex:1}.ambulance-status span:not(.fleet-pulse){font:8px var(--mono);letter-spacing:1px;color:#8ea4b8}.ambulance-status strong{font:10px var(--mono);color:#eaf5fe}.fleet-pulse{width:6px;height:6px;border-radius:50%;background:var(--green);box-shadow:0 0 0 4px rgba(79,224,178,.08)}
.agent-panel{flex:1;min-height:255px}.agent-state-line{display:flex;align-items:center;gap:8px;margin:0 17px 12px;padding:9px 10px;border-radius:7px;border:1px solid rgba(91,231,220,.13);background:rgba(91,231,220,.035);color:var(--cyan);font:8px var(--mono);letter-spacing:.7px}.day-tag{margin-left:auto;border:1px solid var(--border);border-radius:4px;padding:4px 6px;color:#8ea4b8;font-size:8px}.decision-card{margin:0 17px;padding:13px 13px 12px;background:linear-gradient(130deg,rgba(91,231,220,.05),rgba(255,255,255,.012));border:1px solid rgba(91,231,220,.12);border-radius:8px}.decision-label{font:8px var(--mono);letter-spacing:1.2px;color:#71879d;margin-bottom:7px}.decision-title{font-size:13px;font-weight:800;line-height:1.45;color:#f0f7fd}.decision-card p{font-size:10px;line-height:1.65;color:#91a6ba;margin:6px 0 0}.decision-foot{display:grid;grid-template-columns:1fr 1fr;margin:12px 17px 0;border-bottom:1px solid rgba(155,184,211,.09)}.decision-foot div{display:flex;flex-direction:column;gap:5px;padding:0 0 11px}.decision-foot div+div{border-left:1px solid var(--border);padding-left:14px}.decision-foot span{font:8px var(--mono);letter-spacing:1px;color:#6f8499}.decision-foot strong{font:10px var(--mono);color:#dce8f5;white-space:normal;word-break:break-word}.activity-title{display:flex;justify-content:space-between;align-items:center;padding:13px 17px 8px;font:8px var(--mono);letter-spacing:1.15px;color:#8499af}.activity-title span:last-child{color:#60798f}.action-feed{margin:0 17px 14px;max-height:155px;overflow:auto;scrollbar-width:thin;scrollbar-color:#294055 transparent}.feed-empty{font-size:10px;line-height:1.6;color:#71869b;padding:8px 0 1px}.feed-item{display:grid;grid-template-columns:44px minmax(0,1fr);gap:9px;padding:8px 0;border-top:1px solid rgba(155,184,211,.08);animation:feed-enter .3s ease-out}.feed-day{font:9px var(--mono);color:var(--cyan);padding-top:2px}.feed-text{font-size:9px;line-height:1.5;color:#c1d0df}.feed-text small{display:block;margin-top:3px;font-size:8px;color:#788da2}.footer{display:flex;justify-content:space-between;gap:10px;flex-wrap:wrap;padding:14px 2px 3px;color:#60768c;font:8px var(--mono);letter-spacing:.5px}.footer b{font-weight:500;color:#95a9bd}
.leaflet-control-attribution{background:rgba(7,13,22,.82)!important;color:#a0b3c7!important;font-size:9px!important}.leaflet-control-attribution a{color:#7ddbd8!important}.leaflet-control-zoom a{background:#111f2e!important;color:#c5d5e4!important;border-color:rgba(155,184,211,.17)!important}.facility-div-icon,.ambulance-div-icon,.infection-wave-icon,.infectious-halo-icon,.destination-div-icon{background:transparent!important;border:0!important}.facility-emoji{font-size:18px;filter:drop-shadow(0 2px 5px rgba(0,0,0,.8))}.ambulance-shell{position:relative;width:34px;height:32px;display:flex;align-items:center;justify-content:center;filter:drop-shadow(0 2px 5px rgba(0,0,0,.8));animation:vehicle-bob .32s ease-in-out infinite alternate}.ambulance-emoji{font-size:26px;line-height:1}.ambulance-beacon{position:absolute;right:4px;top:1px;width:6px;height:5px;border-radius:2px;background:#6df5ff;box-shadow:0 0 7px #36f1ff;animation:beacon-flash .48s steps(2,end) infinite}.ambulance-tag{position:absolute;left:28px;top:5px;white-space:nowrap;padding:3px 5px;border-radius:4px;background:rgba(5,16,28,.88);border:1px solid rgba(91,231,220,.28);color:#dffcff;font:8px var(--mono);letter-spacing:.4px}.facility-emoji-wrap{font-size:20px;filter:drop-shadow(0 2px 5px rgba(0,0,0,.7))}.infection-core{position:absolute;width:9px;height:9px;border-radius:50%;background:#ff6578;border:1.5px solid #fff0f0;box-shadow:0 0 11px 3px rgba(255,72,100,.55)}.infection-core:before,.infection-core:after{content:"";position:absolute;inset:-5px;border:1.5px solid rgba(255,101,120,.85);border-radius:50%;animation:infection-wave 2s ease-out infinite}.infection-core:after{animation-delay:1s}.wave-node{position:absolute;width:16px;height:16px;left:0;top:0;border-radius:50%;border:2px solid rgba(255,101,120,.95);background:rgba(255,101,120,.13);box-shadow:0 0 12px rgba(255,101,120,.35);animation:wave-expand 2.25s ease-out forwards}.wave-node.exposed{border-color:#ffc36f;background:rgba(255,195,111,.13);box-shadow:0 0 12px rgba(255,195,111,.35)}.destination-cross{width:17px;height:17px;position:relative;border:1.5px solid #b6fff7;border-radius:50%;box-shadow:0 0 13px rgba(91,231,220,.55);animation:destination-pulse 1s ease-in-out infinite}.destination-cross:before,.destination-cross:after{content:"";position:absolute;background:#b6fff7;left:50%;top:50%;transform:translate(-50%,-50%)}.destination-cross:before{height:25px;width:1px}.destination-cross:after{height:1px;width:25px}.dispatch-route-glow{stroke:#49e7dc;stroke-opacity:.24;stroke-width:11;stroke-linecap:round;stroke-linejoin:round}.dispatch-route-core{stroke:#78fff3;stroke-opacity:.98;stroke-width:3.2;stroke-dasharray:8 10;stroke-linecap:round;animation:route-flow .5s linear infinite;filter:drop-shadow(0 0 4px rgba(73,231,220,.65))}.leaflet-tooltip.map-tooltip{background:rgba(7,17,29,.95);border:1px solid rgba(91,231,220,.25);color:#dfedfa;font-size:10px;border-radius:6px;box-shadow:0 6px 20px rgba(0,0,0,.25)}.leaflet-tooltip.map-tooltip:before{display:none}
@keyframes spin{to{transform:rotate(360deg)}}@keyframes soft-pulse{0%,100%{opacity:1;box-shadow:0 0 0 0 rgba(91,231,220,.22)}50%{opacity:.75;box-shadow:0 0 0 5px rgba(91,231,220,0)}}@keyframes infection-wave{0%{transform:scale(.55);opacity:.95}100%{transform:scale(2.8);opacity:0}}@keyframes wave-expand{0%{transform:scale(.4);opacity:0}12%{opacity:1}100%{transform:scale(3.2);opacity:0}}@keyframes route-flow{to{stroke-dashoffset:-18}}@keyframes beacon-flash{0%,48%{opacity:1}49%,100%{opacity:.22}}@keyframes vehicle-bob{from{transform:translateY(0)}to{transform:translateY(-2px)}}@keyframes destination-pulse{0%,100%{transform:scale(.92);opacity:.7}50%{transform:scale(1.1);opacity:1}}@keyframes feed-enter{from{opacity:0;transform:translateY(4px)}to{opacity:1;transform:translateY(0)}}
@media(max-width:1260px){.metric-grid{grid-template-columns:repeat(3,minmax(0,1fr))}.workspace{grid-template-columns:minmax(0,1.45fr) minmax(300px,.8fr)}.intro-row{align-items:flex-start}.controls-wrap{gap:8px}.run-controls{flex-wrap:wrap}}
@media(max-width:930px){.intro-row{flex-direction:column}.controls-wrap{width:100%;align-items:flex-start}.workspace{grid-template-columns:1fr}.side-stack{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));align-items:start}.agent-panel{min-height:0}#map-wrap{height:62vh;min-height:470px}.footer span:nth-child(2){order:3;width:100%}}
@media(max-width:620px){.topbar{height:auto;min-height:70px;padding:12px 14px}.brand-lockup{gap:10px}.brand-mark{width:25px;height:25px}.brand-title{font-size:18px}.brand-title span{display:block;margin:3px 0 0;font-size:8px}.brand-copy .eyebrow{font-size:7px;letter-spacing:.8px}.header-status{gap:9px}.connection-pill{padding:7px 8px;font-size:7px}.day-counter{gap:5px;flex-direction:column;align-items:flex-end}.day-counter .eyebrow{font-size:7px}.day-counter strong{font-size:15px}.page-shell{padding-left:10px;padding-right:10px}.intro-row{padding:19px 0 16px;gap:16px}h1{font-size:30px}.intro-copy p{font-size:11px}.run-controls{display:grid;grid-template-columns:1fr 1fr;width:100%;gap:7px}.btn{padding:8px 9px;gap:7px;min-width:0}.btn strong{font-size:9px}.btn small{font-size:8px}.button-symbol{font-size:14px}.secondary-controls{flex-wrap:wrap;justify-content:flex-start}.metric-grid{grid-template-columns:repeat(2,minmax(0,1fr));gap:8px}.metric-card{padding:11px 12px}.metric-value{font-size:24px}.metric-foot{font-size:8px}.panel-heading{padding:14px 13px 11px}.map-heading{align-items:flex-start}.map-heading-right{align-items:flex-end;flex-direction:column;gap:5px}.map-coordinates{display:none}#map-wrap{height:56vh;min-height:400px}.map-glass-top{top:8px;left:8px;right:8px;gap:5px}.legend-chip{font-size:7px;padding:6px}.map-legend{gap:10px 12px;padding:11px 12px;font-size:8px}.map-note{padding:0 12px 11px;font-size:8px}.side-stack{grid-template-columns:1fr}.response-panel{order:1}.agent-panel{order:2}.footer{font-size:7px;line-height:1.7}.footer span:nth-child(2){width:auto;order:initial}}
@media(prefers-reduced-motion:reduce){*,*::before,*::after{animation-duration:.01ms!important;animation-iteration-count:1!important;scroll-behavior:auto!important;transition-duration:.01ms!important}}


Writing /content/epicontrol_web/style.css


In [ ]:
%%writefile /content/epicontrol_web/app.js
(() => {
  'use strict';

  const $ = (id) => document.getElementById(id);
  const COLORS = {0:'#2dd4bf', 1:'#ffc36f', 2:'#ff6578', 3:'#7abaff', 4:'#8190a4'};
  const n0 = (value) => Number(value || 0);
  const format = (value) => Math.round(n0(value)).toLocaleString('en-US');
  const safe = (value) => String(value ?? '').replace(/[&<>"']/g, (c) => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));

  let map = null;
  let boundaryLayer = null;
  let peopleLayer = null;
  let peopleRenderer = null;
  let zoneLayers = new Map();
  let personMarkers = new Map();
  let previousAgents = new Map();
  let motionTargets = new Map();
  let infectionHaloMarkers = new Map();
  let waveMarkers = [];
  let facilityByKey = new Map();
  let ambulanceMarkers = new Map();
  let routeLayers = new Map();
  let destinationMarkers = new Map();
  let animatingAmbulances = new Set();
  let visualAmbulancePositions = new Map();
  let motionFrame = null;
  let lastRevision = -1;
  let lastDispatchEvent = null;
  let currentState = null;
  let polling = false;
  let facilitiesInitialized = false;

  async function api(path, options = {}) {
    const response = await fetch(path, {
      headers: {'Content-Type':'application/json'},
      cache: 'no-store',
      ...options
    });
    const data = await response.json().catch(() => ({}));
    if (!response.ok) throw new Error(data.detail || `Request failed (${response.status})`);
    return data;
  }
  const post = (path, body = {}) => api(path, {method:'POST', body:JSON.stringify(body)});

  function setConnection(text, good = true) {
    const label = $('connection-label');
    label.textContent = text;
    label.style.color = good ? 'var(--cyan)' : 'var(--red)';
  }

  function setMapStatus(text) {
    $('map-status').textContent = String(text || '').toUpperCase();
  }

  function zoneColor(zone) {
    const infected = n0(zone.infected);
    const exposed = n0(zone.exposed);
    if (infected >= 1) return '#ff6578';
    if (exposed >= 1) return '#ffc36f';
    if (n0(zone.vaccinated) > 0) return '#4fe0b2';
    return '#38a8ad';
  }

  function zoneStyle(zone) {
    const active = n0(zone.infected) + n0(zone.exposed);
    const color = zoneColor(zone);
    return {
      color,
      weight: active ? 1.8 : 1,
      opacity: active ? .98 : .58,
      fillColor: color,
      fillOpacity: active ? Math.min(.34, .10 + Math.log1p(active) * .07) : .035
    };
  }

  function makeDivIcon(className, innerHTML, size = [24, 24], anchor = null) {
    return L.divIcon({
      className,
      html: innerHTML,
      iconSize: size,
      iconAnchor: anchor || [size[0] / 2, size[1] / 2]
    });
  }

  async function initMap() {
    if (!window.L) {
      setConnection('MAP LIBRARY BLOCKED', false);
      setMapStatus('Map library could not load');
      $('map-loading').innerHTML = '<span>Map library blocked by browser/network settings.</span>';
      return;
    }

    map = L.map('map', {
      preferCanvas: true,
      zoomControl: false,
      attributionControl: true,
      minZoom: 13,
      maxZoom: 19,
      zoomAnimation: true,
      fadeAnimation: true,
      markerZoomAnimation: true
    });
    L.control.zoom({position:'bottomright'}).addTo(map);

    // OpenStreetMap's standard tile service does not require an API key.
    // A CSS filter in style.css gives the tiles a dark, low-contrast appearance.
    const tiles = L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
      subdomains: 'abc',
      maxZoom: 19,
      keepBuffer: 3,
      updateWhenIdle: true,
      attribution: '&copy; <a href="https://www.openstreetmap.org/copyright" target="_blank" rel="noopener">OpenStreetMap contributors</a>'
    }).addTo(map);

    let tileErrors = 0;
    tiles.on('tileerror', () => {
      tileErrors += 1;
      if (tileErrors >= 3) {
        setMapStatus('Map tiles unavailable — simulation overlays still active');
      }
    });
    tiles.on('load', () => {
      if (boundaryLayer) setMapStatus(currentState?.running ? 'Live outbreak layer active' : 'Monaco map ready');
    });

    // Use one Leaflet Canvas renderer for population dots, for fewer DOM nodes.
    peopleRenderer = L.canvas({padding: .35});
    peopleLayer = L.layerGroup().addTo(map);

    try {
      const data = await api('/api/map-data');
      boundaryLayer = L.geoJSON(data.boundary, {
        style: {
          color:'#70efe4', weight:2.3, opacity:1,
          fillColor:'#1b8d9c', fillOpacity:.035,
          lineCap:'round', lineJoin:'round'
        }
      }).addTo(map);

      L.geoJSON(data.zones, {
        renderer: peopleRenderer,
        style: () => zoneStyle({}),
        onEachFeature: (feature, layer) => {
          const id = Number(feature.properties.zone_id);
          zoneLayers.set(id, layer);
          layer.bindTooltip(`SIMULATION ZONE ${String(id).padStart(2, '0')}`, {
            className:'map-tooltip', sticky:true, direction:'top'
          });
        }
      }).addTo(map);

      map.fitBounds(boundaryLayer.getBounds().pad(.2), {animate:false});
      $('footer-status').textContent = data.boundary_source || 'MONACO ADMINISTRATIVE BOUNDARY';
      setMapStatus('Monaco boundary ready');
    } catch (error) {
      console.error('[EpiControl] Boundary loading failed:', error);
      map.setView([43.7384, 7.4246], 14);
      $('footer-status').textContent = 'SIMPLIFIED MONACO VIEW';
      setMapStatus('Boundary fallback active');
    }

    $('map-loading').classList.add('is-hidden');
    window.setTimeout(() => map && map.invalidateSize({animate:false}), 180);
    window.addEventListener('resize', () => window.setTimeout(() => map && map.invalidateSize({animate:false}), 100));
  }

  function updateZonePolygons(zones) {
    const lookup = new Map((zones || []).map(zone => [Number(zone.zone), zone]));
    for (const [id, layer] of zoneLayers.entries()) {
      const zone = lookup.get(id);
      if (zone) layer.setStyle(zoneStyle(zone));
    }
  }

  function spawnInfectionWave(agent) {
    if (!map || !agent) return;
    // Cap transient wave nodes to preserve frame rate during large outbreaks.
    while (waveMarkers.length >= 46) {
      const oldest = waveMarkers.shift();
      if (oldest && map.hasLayer(oldest)) map.removeLayer(oldest);
    }
    const isExposed = Number(agent.state) === 1;
    const cls = isExposed ? 'wave-node exposed' : 'wave-node';
    const marker = L.marker([agent.lat, agent.lon], {
      icon: makeDivIcon('infection-wave-icon', `<span class="${cls}"></span>`, [16,16]),
      interactive:false,
      keyboard:false,
      zIndexOffset:650
    }).addTo(map);
    waveMarkers.push(marker);
    window.setTimeout(() => {
      if (map && map.hasLayer(marker)) map.removeLayer(marker);
      waveMarkers = waveMarkers.filter(item => item !== marker);
    }, 2350);
  }

  function updateInfectionHalos(agents) {
    if (!map) return;
    const desired = new Map();
    let count = 0;
    for (const agent of agents || []) {
      if (Number(agent.state) !== 2) continue;
      // A representative subset of active cases gets animated halos.
      if ((Number(agent.id) % 7) !== 2 || count >= 32) continue;
      desired.set(Number(agent.id), agent);
      count += 1;
    }

    for (const [id, marker] of infectionHaloMarkers.entries()) {
      if (!desired.has(id)) {
        map.removeLayer(marker);
        infectionHaloMarkers.delete(id);
      }
    }

    for (const [id, agent] of desired.entries()) {
      if (!infectionHaloMarkers.has(id)) {
        const marker = L.marker([agent.lat, agent.lon], {
          icon: makeDivIcon('infectious-halo-icon', '<span class="infection-core"></span>', [9,9]),
          interactive:false,
          keyboard:false,
          zIndexOffset:450
        }).addTo(map);
        infectionHaloMarkers.set(id, marker);
      }
    }
  }

  function scheduleMotionFrame() {
    if (motionFrame !== null) return;
    motionFrame = requestAnimationFrame(animatePopulationFrame);
  }

  function animatePopulationFrame(now) {
    let active = false;
    for (const [id, motion] of motionTargets.entries()) {
      const rawT = Math.min(1, Math.max(0, (now - motion.start) / motion.duration));
      // Smoothstep easing removes the abrupt stop/start look between daily states.
      const t = rawT * rawT * (3 - 2 * rawT);
      const lat = motion.from.lat + (motion.to.lat - motion.from.lat) * t;
      const lon = motion.from.lng + (motion.to.lng - motion.from.lng) * t;
      motion.marker.setLatLng([lat, lon]);
      const halo = infectionHaloMarkers.get(id);
      if (halo) halo.setLatLng([lat, lon]);
      if (rawT >= 1) motionTargets.delete(id);
      else active = true;
    }
    motionFrame = null;
    if (active && motionTargets.size) motionFrame = requestAnimationFrame(animatePopulationFrame);
  }

  function renderPeople(agents) {
    if (!map || !peopleLayer) return;
    const seen = new Set();
    const nextPrevious = new Map();
    const now = performance.now();
    const transitionMs = 880;

    for (const agent of (agents || [])) {
      const id = Number(agent.id);
      const state = Number(agent.state);
      seen.add(id);
      const old = previousAgents.get(id);
      let marker = personMarkers.get(id);
      const color = COLORS[state] || COLORS[0];
      const radius = state === 2 ? 3.8 : (state === 1 ? 3.2 : 2.6);

      if (!marker) {
        marker = L.circleMarker([agent.lat, agent.lon], {
          renderer: peopleRenderer,
          radius,
          color:'#06121e',
          weight:.55,
          opacity:.8,
          fillColor:color,
          fillOpacity:state === 4 ? .42 : .94
        }).addTo(peopleLayer);
        personMarkers.set(id, marker);
      } else {
        marker.setStyle({
          radius,
          fillColor:color,
          fillOpacity:state === 4 ? .42 : .94,
          color:state === 2 ? '#ffe2e7' : '#06121e',
          weight:state === 2 ? .8 : .55
        });
      }

      if (old && old.state !== state && (state === 1 || state === 2)) {
        if (old.state === 0 || (old.state === 1 && state === 2)) spawnInfectionWave(agent);
      } else if (!old && state === 2) {
        // A few initial infections get an entry pulse so the map is alive at load.
        if (id % 5 === 0) spawnInfectionWave(agent);
      }

      const from = marker.getLatLng();
      const to = L.latLng(Number(agent.lat), Number(agent.lon));
      motionTargets.set(id, {marker, from, to, start:now, duration:transitionMs});
      nextPrevious.set(id, {state});
    }

    for (const [id, marker] of personMarkers.entries()) {
      if (!seen.has(id)) {
        peopleLayer.removeLayer(marker);
        personMarkers.delete(id);
        motionTargets.delete(id);
      }
    }
    previousAgents = nextPrevious;
    updateInfectionHalos(agents || []);
    scheduleMotionFrame();
  }

  function renderFacilities(state) {
    if (!map) return;
    if (!facilitiesInitialized) {
      facilitiesInitialized = true;
      for (const hospital of (state.hospitals || [])) {
        const marker = L.marker([hospital.lat, hospital.lon], {
          icon:makeDivIcon('facility-div-icon', '<span class="facility-emoji-wrap">🏥</span>', [26,26])
        }).addTo(map);
        marker.bindTooltip('', {className:'map-tooltip'});
        facilityByKey.set(`hospital-${hospital.id}`, marker);
      }
      for (const shelter of (state.shelters || [])) {
        const marker = L.marker([shelter.lat, shelter.lon], {
          icon:makeDivIcon('facility-div-icon', `<span class="facility-emoji-wrap">${shelter.open ? '🏘️' : '🏠'}</span>`, [26,26])
        }).addTo(map);
        marker.bindTooltip('', {className:'map-tooltip'});
        facilityByKey.set(`shelter-${shelter.id}`, marker);
      }
      for (const ambulance of (state.ambulances || [])) {
        const marker = L.marker([ambulance.lat, ambulance.lon], {
          icon:makeDivIcon('ambulance-div-icon', `<span class="ambulance-shell"><span class="ambulance-emoji">🚑</span><i class="ambulance-beacon"></i><span class="ambulance-tag">MED ${String(ambulance.id).padStart(2,'0')}</span></span>`, [36,34], [17,17]),
          zIndexOffset:1100
        }).addTo(map);
        marker.bindTooltip(`Ambulance ${ambulance.id}`, {className:'map-tooltip'});
        ambulanceMarkers.set(Number(ambulance.id), marker);
      }
    }

    for (const hospital of (state.hospitals || [])) {
      const marker = facilityByKey.get(`hospital-${hospital.id}`);
      if (marker) marker.setTooltipContent(`Hospital ${hospital.id} · ${hospital.occupied}/${hospital.capacity} beds occupied`);
    }
    for (const shelter of (state.shelters || [])) {
      const marker = facilityByKey.get(`shelter-${shelter.id}`);
      if (marker) {
        marker.setIcon(makeDivIcon('facility-div-icon', `<span class="facility-emoji-wrap">${shelter.open ? '🏘️' : '🏠'}</span>`, [26,26]));
        marker.setTooltipContent(`Shelter ${shelter.id} · ${shelter.open ? 'OPEN' : 'CLOSED'} · ${shelter.occupied}/${shelter.capacity} occupied`);
      }
    }
    renderAmbulances(state.ambulances || []);
  }

  function renderAmbulances(ambulances) {
    for (const ambulance of ambulances || []) {
      const id = Number(ambulance.id);
      const marker = ambulanceMarkers.get(id);
      if (!marker || animatingAmbulances.has(id)) continue;
      if (!ambulance.available && visualAmbulancePositions.has(id)) {
        marker.setLatLng(visualAmbulancePositions.get(id));
      } else {
        if (ambulance.available) visualAmbulancePositions.delete(id);
        marker.setLatLng([ambulance.lat, ambulance.lon]);
      }
      const element = marker.getElement();
      if (element) element.style.opacity = ambulance.available ? '1' : '.98';
    }
  }

  async function roadRoute(from, to) {
    const fallback = [[Number(from.lat), Number(from.lon)], [Number(to.lat), Number(to.lon)]];
    const controller = new AbortController();
    const timeout = window.setTimeout(() => controller.abort(), 1250);
    try {
      const url = `https://router.project-osrm.org/route/v1/driving/${from.lon},${from.lat};${to.lon},${to.lat}?overview=full&geometries=geojson&steps=false`;
      const response = await fetch(url, {signal:controller.signal, mode:'cors', cache:'no-store'});
      if (!response.ok) return fallback;
      const data = await response.json();
      const coordinates = data.routes?.[0]?.geometry?.coordinates;
      if (!coordinates || coordinates.length < 2) return fallback;
      return coordinates.map(([lon, lat]) => [Number(lat), Number(lon)]);
    } catch (_) {
      return fallback;
    } finally {
      window.clearTimeout(timeout);
    }
  }

  function pointAlongPath(coords, fraction, lengths, total) {
    if (!coords.length) return [0,0];
    if (coords.length === 1 || total <= 0) return coords[coords.length - 1];
    const distance = Math.max(0, Math.min(1, fraction)) * total;
    for (let i = 1; i < coords.length; i++) {
      if (lengths[i] >= distance) {
        const segment = lengths[i] - lengths[i - 1];
        const t = segment > 0 ? (distance - lengths[i - 1]) / segment : 0;
        return [
          coords[i - 1][0] + (coords[i][0] - coords[i - 1][0]) * t,
          coords[i - 1][1] + (coords[i][1] - coords[i - 1][1]) * t
        ];
      }
    }
    return coords[coords.length - 1];
  }

  async function animateDispatch(dispatch) {
    if (!dispatch || dispatch.event_id === lastDispatchEvent || !map) return;
    lastDispatchEvent = dispatch.event_id;

    const id = Number(dispatch.ambulance_id);
    const marker = ambulanceMarkers.get(id);
    if (!marker) return;

    animatingAmbulances.add(id);
    const from = dispatch.from || marker.getLatLng();
    const to = dispatch.to || from;
    const startPoint = [Number(from.lat), Number(from.lon)];
    const endPoint = [Number(to.lat), Number(to.lon)];
    marker.setLatLng(startPoint);

    // Show movement immediately using a straight-line fallback while road routing resolves.
    const directRoute = [startPoint, endPoint];
    const routePromise = roadRoute(from, to);
    const firstRoute = await Promise.race([
      routePromise,
      new Promise(resolve => window.setTimeout(() => resolve(directRoute), 720))
    ]);
    const coords = (firstRoute && firstRoute.length > 1) ? firstRoute : directRoute;

    const previousLayers = routeLayers.get(id) || [];
    previousLayers.forEach(layer => { if (map.hasLayer(layer)) map.removeLayer(layer); });
    const glow = L.polyline(coords, {
      renderer:L.svg(), className:'dispatch-route-glow', color:'#49e7dc', weight:10, opacity:.8,
      lineCap:'round', lineJoin:'round', interactive:false
    }).addTo(map);
    const core = L.polyline(coords, {
      renderer:L.svg(), className:'dispatch-route-core', color:'#78fff3', weight:3.2, opacity:.98,
      dashArray:'8 10', lineCap:'round', lineJoin:'round', interactive:false
    }).addTo(map);
    routeLayers.set(id, [glow, core]);

    let destination = destinationMarkers.get(id);
    if (destination && map.hasLayer(destination)) map.removeLayer(destination);
    destination = L.marker(endPoint, {
      icon:makeDivIcon('destination-div-icon', '<span class="destination-cross"></span>', [17,17]),
      interactive:false, keyboard:false, zIndexOffset:1000
    }).addTo(map);
    destinationMarkers.set(id, destination);

    const lengths = [0];
    let total = 0;
    for (let i=1; i<coords.length; i++) {
      total += map.distance(coords[i-1], coords[i]);
      lengths.push(total);
    }
    const duration = 5700;
    const started = performance.now();

    await new Promise(resolve => {
      const step = (now) => {
        const linear = Math.min(1, (now - started) / duration);
        const eased = linear * linear * (3 - 2 * linear);
        const point = pointAlongPath(coords, eased, lengths, total);
        marker.setLatLng(point);
        const element = marker.getElement();
        if (element) element.classList.add('vehicle-in-transit');
        if (linear < 1) requestAnimationFrame(step);
        else resolve();
      };
      requestAnimationFrame(step);
    });

    marker.setLatLng(endPoint);
    visualAmbulancePositions.set(id, endPoint);
    animatingAmbulances.delete(id);
    window.setTimeout(() => {
      for (const layer of (routeLayers.get(id) || [])) if (map.hasLayer(layer)) map.removeLayer(layer);
      routeLayers.delete(id);
      const target = destinationMarkers.get(id);
      if (target && map.hasLayer(target)) map.removeLayer(target);
      destinationMarkers.delete(id);
      const el = marker.getElement();
      if (el) el.classList.remove('vehicle-in-transit');
    }, 4200);
  }

  function renderMetrics(state) {
    const metrics = state.metrics || {};
    $('m-population').textContent = format(metrics.population);
    $('m-infected').textContent = format(metrics.infected);
    $('m-exposed').textContent = format(metrics.exposed);
    $('m-recovered').textContent = format(metrics.recovered);
    $('m-dead').textContent = format(metrics.dead);
    $('m-vaccinated').textContent = format(metrics.vaccinated);
    $('m-new-cases').textContent = `${format(metrics.new_exposures)} new today`;
    $('day-number').innerHTML = `${String(metrics.day || 0).padStart(2,'0')} <small>/ ${state.max_days || 60}</small>`;
    $('m-quarantined').textContent = format(metrics.quarantined);
    $('m-vaccine-stock').textContent = format((state.resources || {}).vaccines);
    $('m-budget').textContent = format((state.resources || {}).budget);

    const capacity = Math.max(1, n0(metrics.hospital_capacity));
    const hospitalized = n0(metrics.hospitalized);
    $('hospital-text').textContent = `${format(hospitalized)} / ${format(capacity)}`;
    $('hospital-bar').style.width = `${Math.min(100, 100 * hospitalized / capacity)}%`;
    $('hospital-bar').style.background = (hospitalized / capacity) > .8
      ? 'linear-gradient(90deg,#ffb66d,#ff536d)'
      : 'linear-gradient(90deg,#5be7dc,#3da9cb)';

    const ambulances = state.ambulances || [];
    const available = ambulances.filter(item => item.available).length;
    $('m-ambulances').textContent = `${available} / ${ambulances.length} available`;
    const stageLabels = {
      idle: 'STANDBY',
      pandemic: state.outbreak_ready ? 'OUTBREAK READY' : 'UNCONTROLLED SPREAD',
      agent: 'DQN RESPONSE'
    };
    $('mode-pill').textContent = stageLabels[state.mode] || String(state.mode || 'idle').replace('_',' ').toUpperCase();

    // The intended workflow is: start spread phase, then hand the same state
    // to the agent. The agent button stays available during the spread phase.
    $('start-baseline').disabled = !!state.running;
    $('start-agent').disabled = !(state.mode === 'pandemic' && state.outbreak_ready);
    const agentSmall = $('start-agent').querySelector('small');
    if (agentSmall) {
      agentSmall.textContent = state.mode === 'pandemic'
        ? (state.outbreak_ready ? 'Take over widespread outbreak' : 'Available when citywide spread is ready')
        : (state.mode === 'agent' ? 'DQN currently in control' : 'Start pandemic phase first');
    }
    $('pause-btn').disabled = !state.running;
    $('pause-btn').textContent = state.paused ? '▶  Resume' : 'Ⅱ  Pause';

    if (state.running && state.mode === 'pandemic') {
      setMapStatus(state.paused ? 'Spread phase paused' : 'Uncontrolled spread · no recoveries or interventions');
    } else if (state.outbreak_ready) {
      setMapStatus('Outbreak established · DQN ready for takeover');
    } else if (state.running && state.mode === 'agent') {
      setMapStatus(state.paused ? 'DQN response paused' : 'DQN response active');
    } else {
      setMapStatus(state.status || 'Ready');
    }
  }

  function renderDecision(state) {
    const action = state.last_action || {};
    $('agent-mode-text').textContent = state.mode === 'agent'
      ? 'DQN STRATEGIC CONTROL'
      : (state.mode === 'pandemic'
          ? (state.outbreak_ready ? 'OUTBREAK READY FOR TAKEOVER' : 'UNCONTROLLED SPREAD · NO INTERVENTIONS')
          : 'AGENT STANDING BY');
    $('action-day-tag').textContent = `DAY ${String(action.day || 0).padStart(2,'0')}`;
    $('decision-title').textContent = action.label || 'Awaiting simulation start';
    $('decision-detail').textContent = action.detail || 'Choose a simulation mode to begin.';
    $('last-reward').textContent = action.reward === null || action.reward === undefined
      ? '—' : Number(action.reward).toFixed(2);
    $('action-result').textContent = String(action.result || (state.running ? 'IN PROGRESS' : 'STANDBY')).toUpperCase().slice(0,30);

    const history = (state.action_history || []).slice(-4).reverse();
    $('activity-count').textContent = `${history.length} EVENT${history.length === 1 ? '' : 'S'}`;
    $('action-feed').innerHTML = history.length ? history.map(item =>
      `<div class="feed-item"><div class="feed-day">D${String(item.day).padStart(2,'0')}</div><div class="feed-text">${safe(item.label)}<small>${safe(item.result || '')}</small></div></div>`
    ).join('') : '<div class="feed-empty">No interventions recorded yet. Agent decisions will appear here as the simulation runs.</div>';
  }

  function renderState(state) {
    currentState = state;
    renderMetrics(state);
    updateZonePolygons(state.zones || []);
    renderPeople(state.agents || []);
    renderFacilities(state);
    renderDecision(state);
    if (state.dispatch_event) animateDispatch(state.dispatch_event);

    if (state.error) {
      setConnection('SIMULATION ERROR', false);
      console.error('[EpiControl] Backend error:', state.error);
    } else if (state.running) {
      if (state.mode === 'pandemic') {
        setConnection(state.paused ? 'SPREAD PAUSED' : 'OUTBREAK SPREADING', true);
      } else {
        setConnection(state.paused ? 'DQN PAUSED' : 'DQN RESPONSE LIVE', true);
      }
    } else if (state.outbreak_ready) {
      setConnection('DQN TAKEOVER READY', true);
    } else {
      setConnection(state.status && state.status.toLowerCase().includes('complete') ? 'EPISODE COMPLETE' : 'SYSTEM READY', true);
    }
  }

  async function pollState() {
    if (polling) return;
    polling = true;
    try {
      const state = await api('/api/state');
      if (state.revision !== lastRevision || !currentState) {
        renderState(state);
        lastRevision = state.revision;
      }
    } catch (error) {
      setConnection('RECONNECTING', false);
      console.warn('[EpiControl] State poll failed:', error);
    } finally {
      polling = false;
    }
  }

  async function startSimulation(mode) {
    try {
      setConnection('STARTING SIMULATION', true);
      await post('/api/start', {mode});
      await pollState();
    } catch (error) {
      setConnection('START FAILED', false);
      window.alert(`Could not start simulation: ${error.message}`);
    }
  }

  $('start-baseline').addEventListener('click', () => startSimulation('pandemic'));
  $('start-agent').addEventListener('click', () => startSimulation('agent'));
  $('pause-btn').addEventListener('click', async () => {
    try { await post('/api/pause'); await pollState(); }
    catch (error) { window.alert(error.message); }
  });
  $('reset-btn').addEventListener('click', async () => {
    try {
      await post('/api/reset');
      lastDispatchEvent = null;
      visualAmbulancePositions.clear();
      animatingAmbulances.clear();
      for (const layers of routeLayers.values()) layers.forEach(layer => map && map.hasLayer(layer) && map.removeLayer(layer));
      routeLayers.clear();
      for (const marker of destinationMarkers.values()) map && map.hasLayer(marker) && map.removeLayer(marker);
      destinationMarkers.clear();
      previousAgents.clear();
      motionTargets.clear();
      await pollState();
    } catch (error) { window.alert(error.message); }
  });
  $('speed-select').addEventListener('change', async (event) => {
    try { await post('/api/speed', {value:Number(event.target.value)}); await pollState(); }
    catch (error) { console.warn('[EpiControl] Could not change speed:', error); }
  });

  (async () => {
    setConnection('CONNECTING', true);
    await initMap();
    await pollState();
    window.setInterval(pollState, 350);
  })();
})();


Writing /content/epicontrol_web/app.js


In [ ]:
# EpiControl FastAPI backend
from pathlib import Path
from threading import Lock, Thread
import time
import uuid
import requests
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from shapely.geometry import shape, mapping, box, Point, Polygon
from shapely.ops import unary_union, nearest_points

WEB_ROOT = Path('/content/epicontrol_web')
WEB_ROOT.mkdir(parents=True, exist_ok=True)

# The environment and model are constructed/loaded by earlier notebook cells.
if 'StrategicEpiControlEnv' not in globals():
    raise RuntimeError('StrategicEpiControlEnv is not defined. Run the V5.6 environment setup cells first.')
if 'dqn_model' not in globals() or not hasattr(dqn_model, 'predict'):
    raise RuntimeError('dqn_model is not loaded. Run the model-loading cell before starting the web app.')
if int(getattr(dqn_model.action_space, 'n', -1)) != 5:
    raise RuntimeError('The loaded model is not the V5.6 strategic DQN with 5 discrete actions.')

MODEL = dqn_model
MAX_DAYS = int(globals().get('SIMULATION_DAYS', 60))
CITY_SIZE_LOCAL = float(globals().get('CITY_SIZE', 100.0))
GRID_SIZE_LOCAL = int(globals().get('GRID_SIZE', 5))
POPULATION_LOCAL = int(globals().get('POPULATION', 1000))


def _fallback_monaco_polygon():
    # Only used if the live geoBoundaries request is unavailable.
    return Polygon([
        (7.4090, 43.7290), (7.4102, 43.7388), (7.4155, 43.7473),
        (7.4258, 43.7508), (7.4359, 43.7464), (7.4391, 43.7392),
        (7.4340, 43.7318), (7.4252, 43.7274), (7.4160, 43.7272),
        (7.4090, 43.7290)
    ])


def _load_monaco_boundary():
    try:
        metadata_url = 'https://www.geoboundaries.org/api/current/gbOpen/MCO/ADM0/'
        meta_response = requests.get(metadata_url, timeout=12)
        meta_response.raise_for_status()
        metadata = meta_response.json()
        geojson_url = metadata.get('simplifiedGeometryGeoJSON') or metadata.get('gjDownloadURL')
        if not geojson_url:
            raise ValueError('geoBoundaries metadata did not provide a GeoJSON URL')
        geo_response = requests.get(geojson_url, timeout=18)
        geo_response.raise_for_status()
        document = geo_response.json()
        if document.get('type') == 'FeatureCollection':
            geometries = [shape(feature['geometry']) for feature in document.get('features', []) if feature.get('geometry')]
            geom = unary_union(geometries)
        elif document.get('type') == 'Feature':
            geom = shape(document['geometry'])
        else:
            geom = shape(document)
        if geom.is_empty or not geom.is_valid:
            geom = geom.buffer(0)
        if geom.is_empty:
            raise ValueError('Downloaded Monaco boundary is empty')
        return geom, 'geoBoundaries gbOpen / MCO ADM0'
    except Exception as exc:
        print(f'Warning: using simplified Monaco outline because geoBoundaries could not be loaded: {exc}')
        return _fallback_monaco_polygon(), 'Simplified Monaco outline fallback'


MONACO_GEOM, BOUNDARY_SOURCE = _load_monaco_boundary()
MIN_LON, MIN_LAT, MAX_LON, MAX_LAT = MONACO_GEOM.bounds
LON_SPAN = max(MAX_LON - MIN_LON, 0.0001)
LAT_SPAN = max(MAX_LAT - MIN_LAT, 0.0001)
BOUNDARY_FEATURE = {
    'type': 'Feature',
    'properties': {'name': 'Monaco', 'source': BOUNDARY_SOURCE},
    'geometry': mapping(MONACO_GEOM.simplify(0.000008, preserve_topology=True))
}


def _to_float(value):
    return float(np.asarray(value).item())


def _visual_geo_point(x, y):
    """Project synthetic city coordinates into Monaco, clamping outside points to the boundary.

    The underlying model remains a synthetic 100x100 city. These coordinates are for visualization only.
    """
    lon = MIN_LON + (float(x) / CITY_SIZE_LOCAL) * LON_SPAN
    lat = MIN_LAT + (float(y) / CITY_SIZE_LOCAL) * LAT_SPAN
    point = Point(lon, lat)
    if not MONACO_GEOM.covers(point):
        point = nearest_points(MONACO_GEOM, point)[0]
    return {'lat': round(float(point.y), 7), 'lon': round(float(point.x), 7)}


def _make_zone_geojson():
    features = []
    step_x = LON_SPAN / GRID_SIZE_LOCAL
    step_y = LAT_SPAN / GRID_SIZE_LOCAL
    for row in range(GRID_SIZE_LOCAL):
        for col in range(GRID_SIZE_LOCAL):
            zone_id = row * GRID_SIZE_LOCAL + col
            cell = box(
                MIN_LON + col * step_x,
                MIN_LAT + row * step_y,
                MIN_LON + (col + 1) * step_x,
                MIN_LAT + (row + 1) * step_y,
            )
            clipped = cell.intersection(MONACO_GEOM)
            if clipped.is_empty:
                continue
            features.append({
                'type': 'Feature',
                'properties': {'zone_id': int(zone_id)},
                'geometry': mapping(clipped.simplify(0.000004, preserve_topology=True))
            })
    return {'type': 'FeatureCollection', 'features': features}


ZONE_GEOJSON = _make_zone_geojson()


def _metric_snapshot(env, daily=None):
    if daily is None:
        daily = {'new_exposures': 0, 'deaths_today': 0, 'recoveries_today': 0}
    result = env._get_metrics(daily)
    result = {k: (int(v) if isinstance(v, (np.integer, bool)) else float(v) if isinstance(v, np.floating) else v) for k, v in result.items()}
    result['hospital_capacity'] = int(env.hospitals['capacity'].sum()) if len(env.hospitals) else int(globals().get('HOSPITAL_CAPACITY', 100))
    return result



def _action_label(env, action, mode):
    if mode == 'pandemic':
        return 'Baseline — no intervention'
    decoded = env.decode_action(int(action))
    kind = decoded.get('type')
    if kind == globals().get('NO_ACTION', 'NO_ACTION'):
        return 'Observe / no intervention'
    if kind == globals().get('VACCINATE', 'VACCINATE'):
        return f"Vaccinate Zone {int(decoded.get('zone', 0)):02d} · up to {int(decoded.get('amount', 25))} people"
    if kind == globals().get('OPEN_SHELTER', 'OPEN_SHELTER'):
        return f"Open emergency shelter {int(decoded.get('shelter_id', 0))}"
    if kind == globals().get('QUARANTINE', 'QUARANTINE'):
        return f"Quarantine Zone {int(decoded.get('zone', 0)):02d} · shelter {int(decoded.get('shelter_id', 0))}"
    if kind == globals().get('DISPATCH_AMBULANCE', 'DISPATCH_AMBULANCE'):
        return f"Dispatch ambulance to Zone {int(decoded.get('zone', 0)):02d}"
    return str(decoded)


def _result_text(action_result):
    if not action_result:
        return 'No intervention taken yet'
    kind = str(action_result.get('type', ''))
    success = bool(action_result.get('success', False))
    if kind == str(globals().get('NO_ACTION', 'NO_ACTION')):
        return 'Baseline day advanced without intervention'
    if kind == str(globals().get('VACCINATE', 'VACCINATE')):
        n = int(action_result.get('amount_delivered', 0))
        return f'{n} people vaccinated' if success else 'Vaccination unavailable — no doses delivered'
    if kind == str(globals().get('OPEN_SHELTER', 'OPEN_SHELTER')):
        return f"Shelter {action_result.get('shelter_id')} opened" if success else 'Shelter could not be opened'
    if kind == str(globals().get('QUARANTINE', 'QUARANTINE')):
        n = int(action_result.get('quarantined', 0))
        return f'{n} people quarantined' if success else 'Quarantine unavailable — no capacity/resources'
    if kind == str(globals().get('DISPATCH_AMBULANCE', 'DISPATCH_AMBULANCE')):
        details = action_result.get('details') or {}
        if success:
            return f"Ambulance {details.get('ambulance_id', '—')} dispatched to hospital {details.get('hospital_id', '—')}"
        return 'Dispatch unsuccessful — no eligible patient or ambulance'
    return 'Action processed' if success else 'Action failed'


SIM_ENV = StrategicEpiControlEnv(max_days=MAX_DAYS)
INITIAL_OBSERVATION, _initial_info = SIM_ENV.reset(seed=42)
_initial_metrics = _metric_snapshot(SIM_ENV)
RUNTIME_LOCK = Lock()
RUNTIME = {
    'env': SIM_ENV,
    'observation': INITIAL_OBSERVATION,
    'running': False,
    'paused': False,
    'mode': 'idle',
    'outbreak_ready': False,
    'display_day_offset': 0,
    'status': 'Ready — choose a simulation mode',
    'error': None,
    'revision': 1,
    'generation': 0,
    'speed': 1.0,
    'current_metrics': _initial_metrics,
    'history': [_initial_metrics.copy()],
    'last_action': {'day': 0, 'label': 'Awaiting simulation start', 'detail': 'Choose START PANDEMIC for a no-intervention baseline or START DQN AGENT to let the trained policy control interventions.', 'result': 'standby', 'reward': None},
    'action_history': [],
    'dispatch_event': None,
    'worker': None,
}


class StartRequest(BaseModel):
    mode: str = Field(default='agent')


class SpeedRequest(BaseModel):
    value: float = Field(default=1.0, ge=0.25, le=4.0)


app = FastAPI(title='EpiControl Live API', docs_url=None, redoc_url=None)
app.mount('/static', StaticFiles(directory=str(WEB_ROOT)), name='static')


@app.get('/')
def home():
    return FileResponse(str(WEB_ROOT / 'index.html'))


@app.get('/api/map-data')
def map_data():
    return {
        'boundary': BOUNDARY_FEATURE,
        'zones': ZONE_GEOJSON,
        'bounds': [[MIN_LAT, MIN_LON], [MAX_LAT, MAX_LON]],
        'boundary_source': BOUNDARY_SOURCE,
    }


def _build_state_payload():
    env = RUNTIME['env']
    metrics = dict(RUNTIME['current_metrics'])
    city = env._get_city_state()
    zones = []
    for row in city.to_dict(orient='records'):
        zone_id = int(row['zone'])
        col, grid_row = zone_id % GRID_SIZE_LOCAL, zone_id // GRID_SIZE_LOCAL
        center = _visual_geo_point((col + 0.5) * CITY_SIZE_LOCAL / GRID_SIZE_LOCAL,
                                   (grid_row + 0.5) * CITY_SIZE_LOCAL / GRID_SIZE_LOCAL)
        zones.append({
            'zone': zone_id,
            'lat': center['lat'], 'lon': center['lon'],
            'population': int(row.get('population', 0)),
            'susceptible': int(row.get('susceptible', 0)),
            'exposed': int(row.get('exposed', 0)),
            'infected': int(row.get('infected', 0)),
            'recovered': int(row.get('recovered', 0)),
            'dead': int(row.get('dead', 0)),
            'vaccinated': int(row.get('vaccinated', 0)),
            'quarantined': int(row.get('quarantined', 0)),
            'hospitalized': int(row.get('hospitalized', 0)),
        })

    states = np.asarray(env.states)
    n_people = int(len(states))
    sample_n = min(420, n_people)
    sample_ids = np.linspace(0, max(n_people - 1, 0), sample_n, dtype=int) if sample_n else []
    agents = []
    for idx in sample_ids:
        location = _visual_geo_point(env.positions[idx, 0], env.positions[idx, 1])
        agents.append({
            'id': int(idx), 'lat': location['lat'], 'lon': location['lon'],
            'state': int(states[idx]),
            'vaccinated': bool(env.vaccinated[idx]),
            'quarantined': bool(env.quarantined[idx]),
            'hospitalized': bool(env.hospitalized[idx]),
        })

    hospitals = []
    for _, h in env.hospitals.iterrows():
        location = _visual_geo_point(h['x'], h['y'])
        hospitals.append({'id': int(h['id']), 'lat': location['lat'], 'lon': location['lon'], 'capacity': int(h['capacity']), 'occupied': int(h['occupied'])})
    shelters = []
    for _, sh in env.shelters.iterrows():
        location = _visual_geo_point(sh['x'], sh['y'])
        shelters.append({'id': int(sh['id']), 'lat': location['lat'], 'lon': location['lon'], 'capacity': int(sh['capacity']), 'occupied': int(sh['occupied']), 'open': bool(sh['open'])})
    ambulances = []
    for _, a in env.ambulances.iterrows():
        location = _visual_geo_point(a['x'], a['y'])
        ambulances.append({'id': int(a['id']), 'lat': location['lat'], 'lon': location['lon'], 'available': bool(a['available'])})

    return {
        'revision': int(RUNTIME['revision']),
        'running': bool(RUNTIME['running']),
        'paused': bool(RUNTIME['paused']),
        'mode': RUNTIME['mode'],
        'stage': ('spread' if RUNTIME['mode'] == 'pandemic' and RUNTIME['running'] else ('ready' if RUNTIME.get('outbreak_ready', False) else ('response' if RUNTIME['mode'] == 'agent' else 'idle'))),
        'outbreak_ready': bool(RUNTIME.get('outbreak_ready', False)),
        'status': RUNTIME['status'],
        'error': RUNTIME['error'],
        'day': int(metrics.get('day', env.current_day)),
        'max_days': int(RUNTIME.get('display_day_offset', 0) + env.max_days),
        'metrics': metrics,
        'resources': {k: int(v) for k, v in env.resources.items()},
        'zones': zones,
        'agents': agents,
        'hospitals': hospitals,
        'shelters': shelters,
        'ambulances': ambulances,
        'history': [{k: (int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v, (np.floating,)) else v) for k, v in item.items()} for item in RUNTIME['history'][-MAX_DAYS-1:]],
        'last_action': dict(RUNTIME['last_action']),
        'action_history': [dict(item) for item in RUNTIME['action_history'][-30:]],
        'dispatch_event': dict(RUNTIME['dispatch_event']) if RUNTIME['dispatch_event'] else None,
        'speed': float(RUNTIME['speed']),
    }


_STATE_PAYLOAD_CACHE = {'revision': -1, 'payload': None}


@app.get('/api/state')
def api_state():
    with RUNTIME_LOCK:
        revision = int(RUNTIME['revision'])
        if _STATE_PAYLOAD_CACHE['revision'] != revision or _STATE_PAYLOAD_CACHE['payload'] is None:
            _STATE_PAYLOAD_CACHE['payload'] = _build_state_payload()
            _STATE_PAYLOAD_CACHE['revision'] = revision
        return _STATE_PAYLOAD_CACHE['payload']


def _set_initial_runtime(seed=42):
    env = RUNTIME['env']
    obs, _info = env.reset(seed=int(seed))
    metrics = _metric_snapshot(env)
    RUNTIME['observation'] = obs
    RUNTIME['running'] = False
    RUNTIME['paused'] = False
    RUNTIME['mode'] = 'idle'
    RUNTIME['outbreak_ready'] = False
    RUNTIME['display_day_offset'] = 0
    RUNTIME['status'] = 'Ready — start the uncontrolled pandemic phase first'
    RUNTIME['error'] = None
    RUNTIME['current_metrics'] = metrics
    RUNTIME['history'] = [metrics.copy()]
    RUNTIME['last_action'] = {'day': 0, 'label': 'Awaiting simulation start', 'detail': 'Choose START PANDEMIC for a no-intervention baseline or START DQN AGENT to let the trained policy control interventions.', 'result': 'standby', 'reward': None}
    RUNTIME['action_history'] = []
    RUNTIME['dispatch_event'] = None
    RUNTIME['revision'] += 1


def _advance_spread_only_day(env):
    """Advance transmission only: no recoveries, deaths, vaccination or care."""
    reference_state = env._get_city_state().copy()
    env.current_day += 1

    # Keep normal synthetic mobility and infection transmission.
    env.positions = move_population(
        env.positions,
        env.states,
        env.hospitalized,
        env.quarantined,
        env.mobility * SPREAD_MOBILITY_MULTIPLIER,
        env.np_random
    )
    env.person_zones = update_person_zones(env.positions)


    original_transmission = float(TRANSMISSION_PROBABILITY)
    original_radius = float(INFECTION_RADIUS)
    try:
        globals()['TRANSMISSION_PROBABILITY'] = min(
            0.90, original_transmission * SPREAD_TRANSMISSION_MULTIPLIER
        )
        globals()['INFECTION_RADIUS'] = original_radius * SPREAD_RADIUS_MULTIPLIER
        env.states, newly_exposed_indices, new_exposures = spread_infection(
            env.positions,
            env.states,
            env.vaccinated,
            env.quarantined,
            env.np_random
        )
    finally:
        globals()['TRANSMISSION_PROBABILITY'] = original_transmission
        globals()['INFECTION_RADIUS'] = original_radius
    if len(newly_exposed_indices):
        env.exposure_day[newly_exposed_indices] = env.current_day

    # Let exposed people become infectious, but deliberately suspend the
    # recovery/death clock until the DQN takes over. This is a staged demo
    # scenario, not a change to the trained environment used for evaluation.
    exposed_indices = np.flatnonzero(env.states == EXPOSED)
    for idx in exposed_indices:
        if env.current_day - int(env.exposure_day[idx]) >= INCUBATION_DAYS:
            env.states[idx] = INFECTED
            env.needs_hospital[idx] = (
                env.np_random.random() < hospitalization_probability(env.ages[idx])
            )

    daily = {
        'new_exposures': int(new_exposures),
        'deaths_today': 0,
        'recoveries_today': 0,
    }
    metrics = env._get_metrics(daily)
    metrics['hospital_capacity'] = int(env.hospitals['capacity'].sum()) if len(env.hospitals) else int(globals().get('HOSPITAL_CAPACITY', 100))

    observation = env._get_observation(reference_state=reference_state)
    env.previous_zone_state = env._get_city_state().copy()
    return observation, metrics, daily


def _outbreak_coverage(env):
    """Return share affected and count of zones showing exposure/infection."""
    affected = int(np.sum((env.states == EXPOSED) | (env.states == INFECTED)))
    population = max(int(len(env.states)), 1)
    city = env._get_city_state()
    active_zones = int(np.sum((city['exposed'] + city['infected']) > 0))
    return affected / population, active_zones, affected


# The outbreak build-up runs as a visible setup phase. At handoff, the internal
# DQN clock is rebased, so the controller gets its full trained response horizon.
SPREAD_PHASE_MAX_DAYS = max(18, min(48, MAX_DAYS - 10))
SPREAD_MOBILITY_MULTIPLIER = 2.4
SPREAD_TRANSMISSION_MULTIPLIER = 2.0
SPREAD_RADIUS_MULTIPLIER = 1.2
SPREAD_TARGET_FRACTION = 0.85
SPREAD_TARGET_ZONE_FRACTION = 0.80


def _outbreak_is_widespread(env):
    affected_fraction, active_zones, affected = _outbreak_coverage(env)
    zone_target = max(1, int(np.ceil(GRID_SIZE_LOCAL * GRID_SIZE_LOCAL * SPREAD_TARGET_ZONE_FRACTION)))
    # Either widespread transmission across most of the city, or near-total
    # city saturation, is enough to hand control to the DQN.
    return (
        affected_fraction >= SPREAD_TARGET_FRACTION and active_zones >= zone_target
    ) or affected_fraction >= 0.94


def _worker_loop(generation):
    try:
        while True:
            phase_finished = False
            terminated = False
            truncated = False
            with RUNTIME_LOCK:
                if generation != RUNTIME['generation'] or not RUNTIME['running']:
                    return
                if RUNTIME['paused']:
                    should_step = False
                    delay = 0.25
                else:
                    should_step = True
                    delay = max(0.25, 0.80 / max(float(RUNTIME['speed']), 0.25))
                    env = RUNTIME['env']
                    obs = RUNTIME['observation']
                    mode = RUNTIME['mode']

                    if mode == 'pandemic':
                        # PHASE 1: visible, uncontrolled spread only. No agent,
                        # recoveries, deaths, vaccinations, quarantine, or care.
                        next_obs, metrics, daily = _advance_spread_only_day(env)
                        RUNTIME['observation'] = next_obs
                        RUNTIME['current_metrics'] = dict(metrics)
                        RUNTIME['history'].append(dict(metrics))
                        affected_fraction, active_zones, affected = _outbreak_coverage(env)
                        RUNTIME['last_action'] = {
                            'day': int(metrics.get('day', env.current_day)),
                            'label': 'Uncontrolled outbreak spreading',
                            'detail': (
                                f"{int(daily['new_exposures'])} new exposures today · "
                                f"{affected_fraction:.0%} of the population infected or exposed · "
                                f"{active_zones}/{GRID_SIZE_LOCAL * GRID_SIZE_LOCAL} zones affected. "
                                "No recoveries, deaths, or interventions occur before DQN takeover."
                            ),
                            'result': f"+{int(daily['new_exposures'])} exposed · 0 deaths · 0 recoveries",
                            'reward': None,
                        }
                        reached_target = _outbreak_is_widespread(env)
                        reached_time_limit = int(metrics.get('day', env.current_day)) >= SPREAD_PHASE_MAX_DAYS
                        RUNTIME['revision'] += 1

                        if reached_target or reached_time_limit:
                            RUNTIME['running'] = False
                            RUNTIME['paused'] = False
                            RUNTIME['outbreak_ready'] = True
                            if reached_target:
                                RUNTIME['status'] = (
                                    f"Outbreak established — {affected_fraction:.0%} of the population affected "
                                    f"across {active_zones}/{GRID_SIZE_LOCAL * GRID_SIZE_LOCAL} zones. "
                                    "Start DQN Agent to continue this exact outbreak."
                                )
                            else:
                                RUNTIME['status'] = (
                                    f"Spread phase reached Day {env.current_day} — "
                                    f"{affected_fraction:.0%} affected across {active_zones}/"
                                    f"{GRID_SIZE_LOCAL * GRID_SIZE_LOCAL} zones. "
                                    "Start DQN Agent to continue this exact state."
                                )
                            RUNTIME['revision'] += 1
                            phase_finished = True
                    else:
                        # PHASE 2: run the trained policy on the SAME environment,
                        # using the observation produced by the spread-only phase.
                        action_value, _state = MODEL.predict(obs, deterministic=True)
                        action = int(np.asarray(action_value).reshape(-1)[0])
                        action_label = _action_label(env, action, 'agent')
                        ambulance_sources = {}
                        for _, amb in env.ambulances.iterrows():
                            ambulance_sources[int(amb['id'])] = (float(amb['x']), float(amb['y']))
                        next_obs, reward, terminated, truncated, info = env.step(action)
                        metrics = dict(info.get('metrics') or _metric_snapshot(env))
                        # Internal DQN day is rebased at takeover; public scenario day remains continuous.
                        metrics['day'] = int(RUNTIME.get('display_day_offset', 0) + env.current_day)
                        metrics['hospital_capacity'] = int(env.hospitals['capacity'].sum()) if len(env.hospitals) else int(globals().get('HOSPITAL_CAPACITY', 100))
                        RUNTIME['observation'] = next_obs
                        RUNTIME['current_metrics'] = metrics
                        RUNTIME['history'].append(metrics.copy())
                        action_result = dict(info.get('action_result') or {})
                        result_text = _result_text(action_result)
                        label_detail = f"Day {int(metrics.get('day', env.current_day))}: {result_text}."
                        action_record = {
                            'day': int(metrics.get('day', env.current_day)),
                            'label': action_label,
                            'detail': label_detail,
                            'result': result_text,
                            'reward': float(reward),
                        }
                        RUNTIME['last_action'] = action_record
                        RUNTIME['action_history'].append({
                            'day': action_record['day'],
                            'label': action_label,
                            'result': result_text,
                        })

                        # Emit a visual ambulance dispatch event for the browser animation.
                        details = action_result.get('details') or {}
                        if action_result.get('success') and details and 'ambulance_id' in details and 'patient_id' in details:
                            ambulance_id = int(details['ambulance_id'])
                            patient_id = int(details['patient_id'])
                            source_xy = ambulance_sources.get(ambulance_id, (CITY_SIZE_LOCAL / 2, CITY_SIZE_LOCAL / 2))
                            target_xy = env.positions[patient_id]
                            RUNTIME['dispatch_event'] = {
                                'event_id': f"{generation}-{int(metrics.get('day', env.current_day))}-{uuid.uuid4().hex[:6]}",
                                'ambulance_id': ambulance_id,
                                'from': _visual_geo_point(source_xy[0], source_xy[1]),
                                'to': _visual_geo_point(target_xy[0], target_xy[1]),
                                'day': int(metrics.get('day', env.current_day)),
                            }

                        RUNTIME['revision'] += 1
                        if terminated or truncated:
                            RUNTIME['running'] = False
                            RUNTIME['paused'] = False
                            RUNTIME['status'] = 'Simulation complete — reset to run another episode'
                            RUNTIME['revision'] += 1
            if not should_step:
                time.sleep(delay)
                continue
            if phase_finished or terminated or truncated:
                return
            time.sleep(delay)
    except Exception as exc:
        with RUNTIME_LOCK:
            if generation == RUNTIME['generation']:
                RUNTIME['running'] = False
                RUNTIME['paused'] = False
                RUNTIME['error'] = f'{type(exc).__name__}: {exc}'
                RUNTIME['status'] = 'Simulation error'
                RUNTIME['revision'] += 1
        print('EpiControl simulation worker error:', repr(exc))

@app.post('/api/start')
def start_simulation(payload: StartRequest):
    mode = payload.mode.strip().lower()
    if mode not in {'pandemic', 'agent'}:
        raise HTTPException(status_code=400, detail="mode must be 'pandemic' or 'agent'")

    with RUNTIME_LOCK:
        current_mode = RUNTIME['mode']
        current_day = int(RUNTIME['env'].current_day)

        if mode == 'pandemic':
            # Explicitly starting a new pandemic is the only action that resets
            # the environment to Day 0.
            RUNTIME['generation'] += 1
            generation = RUNTIME['generation']
            _set_initial_runtime(seed=42)
            RUNTIME['mode'] = 'pandemic'
            RUNTIME['outbreak_ready'] = False
            RUNTIME['running'] = True
            RUNTIME['paused'] = False
            RUNTIME['status'] = 'Uncontrolled outbreak spreading — no interventions, deaths, or recoveries in this phase'
            RUNTIME['last_action'] = {
                'day': 0,
                'label': 'Uncontrolled outbreak started',
                'detail': 'The disease will spread naturally through the synthetic population. Recoveries, deaths, and interventions are suspended until the DQN takes over.',
                'result': 'spread phase',
                'reward': None,
            }
        else:
            # Never silently start the agent from Day 0. Wait until the broad
            # outbreak phase is established, then hand the existing population
            # and disease state over to the trained policy.
            if current_mode == 'idle' and current_day == 0:
                raise HTTPException(
                    status_code=409,
                    detail='Start Pandemic first. The DQN button becomes available after the disease has spread across the city.'
                )
            if current_mode == 'pandemic' and not RUNTIME.get('outbreak_ready', False):
                raise HTTPException(
                    status_code=409,
                    detail='Let the uncontrolled spread phase finish first. The DQN will then continue the same outbreak without resetting the population or infection state.'
                )
            if current_mode == 'agent' and current_day >= int(RUNTIME['env'].max_days):
                raise HTTPException(
                    status_code=409,
                    detail='This response episode is complete. Press Reset, then Start Pandemic to create a new outbreak.'
                )

            # Handoff the same environment. Preserve positions, health states,
            # resources, facilities and the already-spread city. The spread
            # phase deliberately suspended clinical outcomes, so restart only
            # the per-person illness clock at takeover to avoid a single burst
            # of immediate recoveries/deaths on the first agent step.
            RUNTIME['generation'] += 1
            generation = RUNTIME['generation']
            env = RUNTIME['env']
            display_offset = int(env.current_day)
            infectious = (env.states == INFECTED)
            incubating = (env.states == EXPOSED)
            env.exposure_day[infectious] = -INCUBATION_DAYS
            env.exposure_day[incubating] = 0
            env.current_day = 0
            env.max_days = MAX_DAYS
            RUNTIME['display_day_offset'] = display_offset
            RUNTIME['current_metrics']['day'] = display_offset
            RUNTIME['history'][-1]['day'] = display_offset

            # Observation feature -2 is day / max_days. Rebase that one feature
            # to the beginning of the response horizon while preserving the
            # outbreak-derived features and zone trends in the observation.
            observation = np.asarray(RUNTIME['observation'], dtype=np.float32).copy()
            if observation.size >= 2:
                observation[-2] = 0.0
            RUNTIME['observation'] = observation

            RUNTIME['mode'] = 'agent'
            RUNTIME['outbreak_ready'] = False
            RUNTIME['running'] = True
            RUNTIME['paused'] = False
            RUNTIME['error'] = None
            RUNTIME['status'] = f'DQN strategic control running — taking over the citywide outbreak at display Day {display_offset}'
            RUNTIME['last_action'] = {
                'day': display_offset,
                'label': 'DQN taking control of the citywide outbreak',
                'detail': 'The same population, positions, exposed/infected states, facilities, and resources are preserved. The agent now controls interventions; deaths and recoveries can begin during this response phase.',
                'result': 'response phase started · outbreak preserved',
                'reward': None,
            }
            RUNTIME['revision'] += 1

        worker = Thread(target=_worker_loop, args=(generation,), daemon=True, name=f'EpiControlWorker-{generation}')
        RUNTIME['worker'] = worker
        worker.start()
    return {'ok': True, 'mode': mode, 'generation': generation, 'continued_existing_state': mode == 'agent'}

@app.post('/api/pause')
def toggle_pause():
    with RUNTIME_LOCK:
        if not RUNTIME['running']:
            return {'ok': True, 'paused': False}
        RUNTIME['paused'] = not RUNTIME['paused']
        RUNTIME['status'] = 'Simulation paused' if RUNTIME['paused'] else ('Baseline epidemic running' if RUNTIME['mode'] == 'pandemic' else 'DQN strategic control running')
        RUNTIME['revision'] += 1
        return {'ok': True, 'paused': RUNTIME['paused']}


@app.post('/api/reset')
def reset_simulation():
    with RUNTIME_LOCK:
        RUNTIME['generation'] += 1
        _set_initial_runtime(seed=42)
    return {'ok': True}


@app.post('/api/stop')
def stop_simulation():
    with RUNTIME_LOCK:
        RUNTIME['generation'] += 1
        RUNTIME['running'] = False
        RUNTIME['paused'] = False
        RUNTIME['status'] = 'Simulation stopped'
        RUNTIME['revision'] += 1
    return {'ok': True}


@app.post('/api/speed')
def set_speed(payload: SpeedRequest):
    with RUNTIME_LOCK:
        RUNTIME['speed'] = float(payload.value)
        RUNTIME['revision'] += 1
        return {'ok': True, 'speed': RUNTIME['speed']}

print('EpiControl FastAPI backend prepared.')
print('Model:', type(MODEL).__name__, '| observation shape:', getattr(MODEL.observation_space, 'shape', None), '| actions:', getattr(MODEL.action_space, 'n', None))
print('Boundary source:', BOUNDARY_SOURCE)
print('Static app directory:', WEB_ROOT)


EpiControl FastAPI backend prepared.
Model: DQN | observation shape: (308,) | actions: 5
Boundary source: geoBoundaries gbOpen / MCO ADM0
Static app directory: /content/epicontrol_web


In [ ]:
# Start FastAPI in the background and create a temporary public HTTPS link.
import os
import re
import sys
import time
import queue
import socket
import stat
import subprocess
from pathlib import Path
from threading import Thread
from IPython.display import display, HTML
import requests
import uvicorn

# Close a previous EpiControl server/tunnel if this cell is rerun.
_old_server = globals().get('epicontrol_server')
_old_server_thread = globals().get('epicontrol_server_thread')
if _old_server is not None:
    try:
        _old_server.should_exit = True
    except Exception:
        pass
if _old_server_thread is not None and _old_server_thread.is_alive():
    _old_server_thread.join(timeout=3)
_old_tunnel = globals().get('epicontrol_tunnel')
if _old_tunnel is not None and _old_tunnel.poll() is None:
    try:
        _old_tunnel.terminate()
        _old_tunnel.wait(timeout=4)
    except Exception:
        try:
            _old_tunnel.kill()
        except Exception:
            pass

# Select an available port so this cell can be rerun without port conflicts.
_port_socket = socket.socket()
_port_socket.bind(('127.0.0.1', 0))
EPICONTROL_PORT = int(_port_socket.getsockname()[1])
_port_socket.close()

_epicontrol_config = uvicorn.Config(app, host='0.0.0.0', port=EPICONTROL_PORT, log_level='warning', access_log=False)
epicontrol_server = uvicorn.Server(_epicontrol_config)
epicontrol_server_thread = Thread(target=epicontrol_server.run, daemon=True, name='EpiControl-HTTP')
epicontrol_server_thread.start()

_local_ready = False
for _ in range(40):
    try:
        _r = requests.get(f'http://127.0.0.1:{EPICONTROL_PORT}/api/state', timeout=2)
        if _r.ok:
            _local_ready = True
            break
    except Exception:
        time.sleep(0.25)
if not _local_ready:
    raise RuntimeError('The EpiControl FastAPI server did not become ready. Check the backend cell output above.')
print(f'EpiControl backend is running on port {EPICONTROL_PORT}.')

# Download cloudflared if it is not already installed in this Colab runtime.
CLOUDFLARED_PATH = Path('/usr/local/bin/cloudflared')
if not CLOUDFLARED_PATH.exists():
    print('Installing cloudflared for a temporary HTTPS link...')
    _download = requests.get(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        timeout=120,
    )
    _download.raise_for_status()
    if len(_download.content) < 1_000_000:
        raise RuntimeError('Cloudflared download was unexpectedly small; check the network and rerun this cell.')
    CLOUDFLARED_PATH.write_bytes(_download.content)
    CLOUDFLARED_PATH.chmod(CLOUDFLARED_PATH.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

# Read tunnel output in a separate thread so the notebook can detect the generated URL.
_tunnel_lines = queue.Queue()

def _read_tunnel_output(process, output_queue):
    try:
        for _line in iter(process.stdout.readline, ''):
            output_queue.put(_line.rstrip())
    except Exception as _read_error:
        output_queue.put(f'[tunnel log reader] {_read_error}')

epicontrol_tunnel = subprocess.Popen(
    [str(CLOUDFLARED_PATH), 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{EPICONTROL_PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
epicontrol_tunnel_reader = Thread(target=_read_tunnel_output, args=(epicontrol_tunnel, _tunnel_lines), daemon=True)
epicontrol_tunnel_reader.start()

EPICONTROL_PUBLIC_URL = None
_tunnel_log_tail = []
_deadline = time.time() + 75
while time.time() < _deadline:
    try:
        _line = _tunnel_lines.get(timeout=1)
        _tunnel_log_tail.append(_line)
        _tunnel_log_tail = _tunnel_log_tail[-18:]
        _match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', _line)
        if _match:
            EPICONTROL_PUBLIC_URL = _match.group(0)
            break
    except queue.Empty:
        if epicontrol_tunnel.poll() is not None:
            break

if not EPICONTROL_PUBLIC_URL:
    print('The web app is running locally, but the temporary tunnel URL was not detected.')
    print('Recent tunnel output:')
    print('\n'.join(_tunnel_log_tail) or '(no tunnel output yet)')
    raise RuntimeError('Cloudflare Quick Tunnel did not return a public URL. Rerun this launch cell or inspect the tunnel logs above.')

print('\nEpiControl is ready in a separate browser tab:')
print(EPICONTROL_PUBLIC_URL)
display(HTML(
    '<div style="padding:16px 18px;margin:10px 0;border-radius:10px;background:#0d1a27;border:1px solid #2ed6cc;">'
    '<div style="color:#8ea6bc;font:12px monospace;letter-spacing:1px;margin-bottom:8px;">EPICONTROL COMMAND CENTER</div>'
    '<a href="' + EPICONTROL_PUBLIC_URL + '" target="_blank" rel="noopener noreferrer" '
    'style="color:#4ce5dc;font-size:19px;font-weight:700;text-decoration:none;">↗ OPEN DASHBOARD IN A NEW TAB</a>'
    '<div style="color:#9bb0c4;font-size:12px;margin-top:8px;">Keep this Colab session and this tunnel cell alive while using the dashboard.</div>'
    '</div>'
))
print('Access note: this temporary link is public to anyone who has it. The simulation uses synthetic data.')


EpiControl backend is running on port 58849.
Installing cloudflared for a temporary HTTPS link...

EpiControl is ready in a separate browser tab:
https://briefs-referenced-amended-exciting.trycloudflare.com


Access note: this temporary link is public to anyone who has it. The simulation uses synthetic data.
